# 3 — Models and results

Reads `data.pkl` + `problem_info.csv` from notebook 1 and produces the paper's
figures and table.

The circuit is a root sum node over R global archetypes, a product node partitioning
skills into disjoint clusters, a per-cluster sum node over L local states, and
per-skill Bernoulli leaves; it is fit on the exact log marginal likelihood by Adam.
The final cells produce what the paper needs: the archetype figures, the transfer
table, and the three-way comparison against the BKT and DKT baselines.

| output | what it is |
| --- | --- |
| `dataset1-cwh-mean.png` | change4: P(local state = high) per archetype x cluster, mean ± sd |
| `dataset1-archetype-distribution.png` | change4: student archetype belief, in log-odds |
| `dataset2-cwh-mean.png` | change3, same |
| `dataset2-archetype-distribution.png` | change3, same |
| *printed* | mean AUC (± sd) for change4's new skills, by opportunity -- the transfer table, printed for reading off rather than written to a file |

## Imports

In [ ]:
from collections import Counter, defaultdict
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import os
import pickle
import time

from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

## Paths

In [ ]:
# Paths. Everything lives inside this folder, so the notebook is self-contained.
NOTEBOOK_DIR = Path.cwd()

DATASET_DIR = Path(os.environ.get("PC_DATASET_DIR", NOTEBOOK_DIR / "dataset"))
OUTPUT_DIR = Path(os.environ.get("PC_OUTPUT_DIR", NOTEBOOK_DIR / "outputs"))
RESULTS_DIR = Path(os.environ.get("PC_RESULTS_DIR", NOTEBOOK_DIR / "results"))


def workspace_dir(workspace, create=False):
    """Output directory holding one workspace's data.pkl and problem_info.csv."""
    path = OUTPUT_DIR / workspace
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path


print(f"dataset: {DATASET_DIR}\noutputs: {OUTPUT_DIR}\nresults: {RESULTS_DIR}")

## Metrics

Shared metrics. Every model in these notebooks is scored through these.

In [ ]:
EPS = 1e-9


def _rankdata_avg(x: np.ndarray) -> np.ndarray:
    """Average ranks (1-indexed), handling ties -- a tie-correct AUC without scipy."""
    sorter = np.argsort(x, kind="mergesort")
    x_sorted = x[sorter]
    n = len(x)
    ranks_sorted = np.empty(n, dtype=float)
    i = 0
    while i < n:
        j = i
        while j < n and x_sorted[j] == x_sorted[i]:
            j += 1
        ranks_sorted[i:j] = (i + j - 1) / 2.0 + 1
        i = j
    ranks = np.empty(n, dtype=float)
    ranks[sorter] = ranks_sorted
    return ranks


def compute_auc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """AUC via the Mann-Whitney U statistic.

    NaN when `y_true` is single-class -- AUC is undefined there, and small or easy skills
    hit that case often enough that returning NaN beats raising.
    """
    y_true = np.asarray(y_true)
    n_pos = int((y_true == 1).sum())
    n_neg = int((y_true == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = _rankdata_avg(np.asarray(y_pred, dtype=float))
    return float((ranks[y_true == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def compute_binary_classification_metrics(y_true: np.ndarray, y_pred_prob: np.ndarray) -> dict:
    """Accuracy (0.5 threshold), AUC, RMSE and log-loss for a probabilistic predictor."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred_prob = np.clip(np.asarray(y_pred_prob, dtype=float), EPS, 1 - EPS)
    return dict(
        n=len(y_true),
        accuracy=float(((y_pred_prob >= 0.5).astype(float) == y_true).mean()),
        auc=compute_auc(y_true, y_pred_prob),
        rmse=float(np.sqrt(np.mean((y_pred_prob - y_true) ** 2))),
        log_loss=float(-np.mean(y_true * np.log(y_pred_prob)
                                + (1 - y_true) * np.log(1 - y_pred_prob))),
    )

## The skill configuration

The skill configuration: which skills are modeled, and which share a cluster.

The only cell naming skills. Everything below the derived line follows from the three
declarations above it. A cluster is a modeling assumption -- skills judged to share an
underlying ability -- not something read off the data.

In [ ]:
# Never modeled: unmapped recognition probes, and the duplicate strategy event.
# `SelectOptimalStrategy` itself IS modeled -- it is the `strategychoice` cluster.
EXCLUDE_SKILLS = {"Recognize-ER", "Recognize-ME", "SelectOptimalStrategyBeforeFA"}

# change3 writes the terminal labeling step with a stray period. Same step -- same slot
# position, same terminal role, comparable accuracy -- so it is canonicalized on kc_list at
# load time, before any table is built. Left alone it would look like a skill new to
# change4 that change3's students had in fact practised heavily. This is the only such pair
# between the two kc_lists.
SKILL_ALIASES = {
    "enter label of final answer.-1": "enter label of final answer-1",
}

# ── 1. The curriculum: cluster -> skills. change4 (dataset-1), the full skill set. ─────
# Cluster names are the labels the figures carry, and `build_skill_index` sorts them, so
# this naming also fixes the column order of the heatmaps.
CURRICULUM = {
    # means and extremes (cross-multiplication) strategy
    "ME": [
        "calculate product of means or extremes-1",
        "calculate solution with means and extremes using difficult numbers-1",
        "calculate solution with means and extremes using simple numbers-1",
        "enter first extreme in equation-1",
        "enter first mean in equation-1",
        "enter second extreme in equation-1",
        "enter second mean in equation-1",
    ],
    # equivalent-ratio (fraction-factor) strategy
    "ER": [
        "calculate part in proportion with fractions-1",
        "calculate total in proportion with fractions-1",
        "enter denominator of form of 1-1 for fractional factor",
        "enter denominator of form of 1-1 for integer factor",
        "enter numerator of form of 1-1 for fractional factor",
        "enter numerator of form of 1-1 for integer factor",
    ],
    # final-answer computation, gated by strategy commitment
    "FinalAnswer": [
        "calculate final amount in context-1",
        "calculate percent change in context-1",
        "calculate part-percent-1",
        "calculate percent-1",
        "calculate total-percent-1",
    ],
    # reading the problem statement into the proportion, in given quantities
    "comprehension": [
        "enter given original amount in proportion-1",
        "enter given total in proportion-1",
        "enter given amount of change in proportion-1",
        "enter given part in proportion-1",
        "enter given percent change in proportion-1",
        "enter numerator of given percent in proportion-1",
        "identify percent change as increase or decrease-1",
    ],
    # the same, but placing the unknown as a variable
    "comprehension(v)": [
        "represent part or total in proportion with variable-1",
        "enter amount of change with variable-1",
        "enter numerator of percent with variable-1",
        "enter numerator of percent change with variable-1",
    ],
    # labeling / interpreting the answer
    "label": [
        "enter label of final answer-1",
        "enter proportion label in numerator-1",
        "enter proportion label in denominator-1",
    ],
    # strategy selection itself
    "strategychoice": [
        "SelectOptimalStrategy",
    ],
}

# ── 2. What change3 (dataset-2) does not model, split by reason. ──────────────────────
# Only the second group is a choice; the transfer experiments read the distinction.
CHANGE3_ABSENT = {          # never in change3's kc_list -- nothing to drop
    "calculate part-percent-1",
    "calculate percent-1",
    "calculate total-percent-1",
    "enter given total in proportion-1",
    "enter given part in proportion-1",
    "enter numerator of given percent in proportion-1",
    "enter proportion label in denominator-1",
    "enter proportion label in numerator-1",
    "enter numerator of percent with variable-1",
    "represent part or total in proportion with variable-1",
}
CHANGE3_EXCLUDED = set()    # present in change3's data, deliberately left unmodeled

# ── 3. Deliberate cluster divergence between the two datasets. ────────────────────────
CLUSTER_OVERRIDES_3 = {}

# ═══ derived -- no skill names below this line ════════════════════════════════════════
_dupes = [s for c in CURRICULUM for s in CURRICULUM[c]
          if sum(s in v for v in CURRICULUM.values()) > 1]
assert not _dupes, f"skill(s) in more than one cluster: {sorted(set(_dupes))}"

# EXCLUDE_SKILLS is the single switch for ablating a skill out of both datasets: the maps
# follow it, so an ablation never leaves a cluster entry pointing at a skill no table has.
SKILL_CLUSTER_MAP = {s: c for c, skills in CURRICULUM.items() for s in skills
                     if s not in EXCLUDE_SKILLS}
_change3_drops = set(CHANGE3_ABSENT) | set(CHANGE3_EXCLUDED)
SKILL_CLUSTER_MAP_3 = {s: CLUSTER_OVERRIDES_3.get(s, c)
                       for s, c in SKILL_CLUSTER_MAP.items() if s not in _change3_drops}

_declared = {s for v in CURRICULUM.values() for s in v}
_stray = _change3_drops - _declared
assert not _stray, f"CHANGE3_ABSENT/EXCLUDED names skill(s) absent from CURRICULUM: {sorted(_stray)}"

# Subsets are the map key sets by construction, so `skills_to_include` means exactly
# "keep what this dataset's cluster map covers".
SKILL_SUBSET = set(SKILL_CLUSTER_MAP)        # change4
SKILL_SUBSET_3 = set(SKILL_CLUSTER_MAP_3)    # change3

# What the transfer experiments read the archetype FROM, and predict zero-shot TO.
COMMON_SKILLS = sorted(set(SKILL_CLUSTER_MAP) & set(SKILL_CLUSTER_MAP_3))
NEW_SKILLS = sorted(set(SKILL_CLUSTER_MAP) - set(SKILL_CLUSTER_MAP_3))

_BY_WORKSPACE = {
    "ratio_proportion_change4": (SKILL_CLUSTER_MAP, SKILL_SUBSET),
    "ratio_proportion_change3": (SKILL_CLUSTER_MAP_3, SKILL_SUBSET_3),
}
_ALIASES = {"change4": "ratio_proportion_change4", "change3": "ratio_proportion_change3"}


def for_workspace(name: str):
    """(cluster map, skill subset) for a workspace, by full name or short alias."""
    key = _ALIASES.get(name, name)
    if key not in _BY_WORKSPACE:
        raise KeyError(f"unknown workspace {name!r}; "
                       f"choose from {sorted(_BY_WORKSPACE) + sorted(_ALIASES)}")
    return _BY_WORKSPACE[key]


def report_skill_coverage(joint_pre_subset, cluster_map, name: str, verbose: bool = True) -> dict:
    """Every skill a workspace observed, and what the model does with it.

    Run on the PRE-subset joint table. `dangling` is the bug worth catching: a cluster-map
    name matching nothing in the data (a typo, or a label that differs between workspaces)
    would otherwise vanish silently.
    """
    counts = joint_pre_subset["skill"].value_counts()
    in_data = set(counts.index)
    mapped = set(cluster_map)
    dangling = sorted(mapped - in_data)
    dropped = sorted(in_data - mapped)

    if verbose:
        print(f"[{name}] {len(in_data)} skills observed, {len(in_data & mapped)} modeled, "
              f"{len(dropped)} dropped")
        if dropped:
            n_lost = int(counts[dropped].sum())
            print(f"          dropped costs {n_lost:,} of {int(counts.sum()):,} "
                  f"opportunities ({n_lost / counts.sum():.1%}):")
            for s in dropped:
                why = "excluded by choice" if s in CHANGE3_EXCLUDED else "not in CURRICULUM"
                print(f"            {counts[s]:>7,}  {s!r}  -- {why}")
        if dangling:
            print(f"  WARNING: {len(dangling)} cluster-map skill(s) match nothing in this "
                  f"dataset -- check for a typo or a differing label:")
            for s in dangling:
                print(f"            {s!r}")
    return dict(dangling=dangling, dropped=dropped)


def check_zero_shot(joint_change3_pre_subset, verbose: bool = True) -> list:
    """The skills transfer treats as new to change4, that change3's students in fact saw.

    Transfer is only zero-shot for skills absent from change3's data. A skill can land in
    NEW_SKILLS for the other reason -- present, but left unmodeled by CHANGE3_EXCLUDED --
    and its score is then neither fair credit nor fair penalty.
    """
    counts = joint_change3_pre_subset["skill"].value_counts()
    practised = sorted(set(NEW_SKILLS) & set(counts.index))
    if verbose and practised:
        print(f"WARNING: {len(practised)} of the {len(NEW_SKILLS)} skills transfer treats "
              f"as new to change4 are present in change3's data:")
        for s in practised:
            print(f"  {counts[s]:>9,} change3 opportunities  {s!r}")
    elif verbose:
        print(f"All {len(NEW_SKILLS)} transfer target skills are absent from change3's "
              f"data -- the zero-shot framing holds.")
    return practised

## Device and dtype

Device and dtype for the two gradient-fit models.

CPU by default: R, L and K are small here, so CPU is usually as fast as MPS and avoids
its memory errors. float64 wherever it is supported -- the PC's recurrence is
multiplicative, and float64's dynamic range makes it markedly more robust; MPS has no
float64 kernels, so that path falls back to float32.

In [ ]:
FORCE_CPU = os.environ.get("PC_FORCE_CPU", "1") != "0"

if FORCE_CPU:
    device = torch.device("cpu")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

dtype = torch.float32 if device.type == "mps" else torch.float64

## The joint opportunity table

data.pkl + problem_info.csv -> the joint opportunity table.

One row per observed skill opportunity:

    student_id, problem_id, problem_order, slot, skill, correct, problem_type,
    time_taken_seconds

`cluster` is attached later by `apply_skill_subset`, once a cluster map is chosen.

In [ ]:
PROBLEM_INFO_COLS = ["school", "student", "count", "problem", "prob_type",
                     "label_opt", "tot_prob_time"]

# Opportunities are capped at each student's first MAX_PROBS_PER_STUDENT problems, matching
# the convention the build pipeline uses.
MAX_PROBS_PER_STUDENT = 20


def build_student_school_map_from_info(prob_info: pd.DataFrame) -> dict:
    """student -> school. Raises rather than taking the first: a student in two schools
    cannot be stratified on, and is a data issue to resolve upstream."""
    counts = prob_info.groupby("student")["school"].nunique()
    bad = counts[counts > 1]
    if not bad.empty:
        raise ValueError(f"{len(bad)} student(s) map to more than one school -- e.g. "
                         f"{bad.index[0]!r}. Resolve this before stratifying by school.")
    return prob_info.groupby("student")["school"].first().to_dict()


def load_workspace(ws_dir, name, max_probs_per_student=MAX_PROBS_PER_STUDENT,
                   exclude_skills=EXCLUDE_SKILLS, skill_aliases=SKILL_ALIASES,
                   all_opt_filter=True, verbose=True) -> dict:
    """Read one workspace's artefacts into a joint table and a student -> school map."""
    t0 = time.time()
    with open(os.path.join(ws_dir, "data.pkl"), "rb") as f:
        ds = pickle.load(f)

    responses = ds["responses"]              # (N, T, Q) int8
    q_template_idx = ds["q_template_idx"]    # (N, T, Q) int
    q_templates = ds["q_templates"]          # (n_templates, K) bool
    response_mask = ds["response_mask"]      # (N, T, Q) bool
    prob_types = ds["prob_types"]            # (N, T) int
    student_ids = np.asarray(ds["student_ids"])
    kc_list = list(ds["kc_list"])
    prob_type_list = list(ds["prob_type_list"])

    # Canonicalize before kc_list is used for anything: the skill column, the coverage
    # report and the transfer skill lists all read the renamed names.
    renamed = [(k, skill_aliases[k]) for k in kc_list if k in skill_aliases]
    if renamed:
        kc_list = [skill_aliases.get(k, k) for k in kc_list]
        collisions = sorted({k for k in kc_list if kc_list.count(k) > 1})
        if verbose:
            for src, dst in renamed:
                print(f"[{name}] alias: {src!r} -> {dst!r}")
            if collisions:
                print(f"[{name}] NOTE: aliasing merged two KC indices onto {collisions} -- "
                      f"their opportunities now pool into one skill.")

    if "problem_names" not in ds:
        raise RuntimeError(f"{ws_dir}/data.pkl is missing 'problem_names'. "
                           f"Re-run notebook 1.")
    problem_names_lol = ds["problem_names"]

    template_to_kc = q_templates.argmax(axis=1)
    n_students, t_max, q_max = responses.shape
    if verbose:
        print(f"[{name}] data.pkl: N={n_students} students, T_max={t_max}, Q_max={q_max}, "
              f"K={len(kc_list)} KCs")

    # (N, T, Q) -> one row per observed opportunity.
    n_idx, t_idx, q_idx = np.nonzero(response_mask)
    keep = t_idx < max_probs_per_student
    n_idx, t_idx, q_idx = n_idx[keep], t_idx[keep], q_idx[keep]

    opp_kc_idx = template_to_kc[q_template_idx[n_idx, t_idx, q_idx]]
    opp_correct = responses[n_idx, t_idx, q_idx].astype(np.int8)
    opp_prob_type = prob_types[n_idx, t_idx]

    opp_problem_name = np.empty(len(n_idx), dtype=object)
    for i, (n, t) in enumerate(zip(n_idx, t_idx)):
        opp_problem_name[i] = problem_names_lol[n][t]

    joint = pd.DataFrame({
        "student_id": student_ids[n_idx],
        "problem_order": t_idx.astype(np.int32),
        "slot": q_idx.astype(np.int16),
        "skill": [kc_list[k] for k in opp_kc_idx],
        "correct": opp_correct,
        "problem_type": [prob_type_list[p] for p in opp_prob_type],
        "problem_id": opp_problem_name,
    })
    # (problem_order, slot) is chronological; mergesort keeps it stable.
    joint = joint.sort_values(["student_id", "problem_order", "slot"],
                              kind="mergesort").reset_index(drop=True)
    if verbose:
        print(f"[{name}] flattened: {joint.shape}")

    joint = joint[~joint["skill"].isin(exclude_skills)].reset_index(drop=True)

    prob_info = pd.read_csv(os.path.join(ws_dir, "problem_info.csv"),
                            usecols=PROBLEM_INFO_COLS)
    prob_info = prob_info[prob_info["count"] <= max_probs_per_student].copy()
    if verbose:
        print(f"[{name}] problem_info.csv (capped): {prob_info.shape}, "
              f"{prob_info['student'].nunique()} students")

    if all_opt_filter:
        chk = prob_info.groupby("student")["label_opt"].apply(lambda x: (x >= 0).all())
        all_opt_students = set(chk[chk].index)
        n_before = joint["student_id"].nunique()
        joint = joint[joint["student_id"].isin(all_opt_students)].reset_index(drop=True)
        if verbose:
            n_after = joint["student_id"].nunique()
            print(f"[{name}] all_opt filter: {n_before} -> {n_after} students "
                  f"({n_after / max(n_before, 1):.1%})")

    student_school_map = build_student_school_map_from_info(prob_info)

    # Problem-level wall clock spread evenly over that problem's opportunities. Kept for
    # schema compatibility; the PC's batching discards this channel before fitting.
    pi = prob_info[["student", "count", "tot_prob_time"]].copy()
    pi["problem_order"] = pi["count"].astype(np.int32) - 1
    pi = pi.rename(columns={"student": "student_id"})
    joint = joint.merge(pi[["student_id", "problem_order", "tot_prob_time"]],
                        on=["student_id", "problem_order"], how="left")
    n_opps = joint.groupby(["student_id", "problem_order"])["skill"].transform("size")
    joint["time_taken_seconds"] = joint["tot_prob_time"] / 1000.0 / n_opps
    joint["time_taken_seconds"] = (joint["time_taken_seconds"]
                                   .fillna(joint["time_taken_seconds"].median())
                                   .clip(lower=0.5))
    joint = joint.drop(columns=["tot_prob_time"])

    if verbose:
        print(f"[{name}] joint ready: {joint.shape}, "
              f"{joint['student_id'].nunique()} students, "
              f"{joint['skill'].nunique()} skills, "
              f"{joint['problem_id'].nunique()} problems  ({time.time() - t0:.1f}s)\n")
    return dict(joint=joint, student_school_map=student_school_map,
                kc_list=kc_list, prob_type_list=prob_type_list)


def apply_skill_subset(joint: pd.DataFrame, skill_cluster_map: dict,
                       skills_to_include=None, name: str = "",
                       verbose: bool = True) -> pd.DataFrame:
    """Drop skills outside the subset entirely, then attach the `cluster` column."""
    joint = joint.copy()
    if skills_to_include is not None:
        n_before = len(joint)
        joint = joint[joint["skill"].isin(skills_to_include)].reset_index(drop=True)
        if verbose:
            print(f"[{name}] skill subset filter: {n_before} -> {len(joint)} rows "
                  f"({joint['skill'].nunique()}/{len(skills_to_include)} of the subset's "
                  f"skills present)")
    joint["cluster"] = joint["skill"].map(skill_cluster_map).fillna("unclustered")
    return joint

## School-stratified splits

School-stratified three-way student splits.

Students, not opportunities, are split. TRAIN fits every model; VAL picks the stopping
epoch for the two gradient-fit models and is used for nothing else; TEST is scored and
never touches fitting or model selection. BKT has no stopping knob and ignores VAL, so all
three models are still compared on identical TEST opportunities.

In [ ]:
SPLIT_NAMES = ("train", "val", "test")


def stratified_school_split(student_ids, student_school_map, train_frac=0.60,
                            val_frac=0.20, seed=0, verbose=True):
    """Sample each school's students independently, so no school lands entirely on one side.

    Schools too small for all three parts fill train -> test -> val: a 1-student school goes
    to train, a 2-student school splits train + test. Test is filled before val because test
    is what gets reported, while val only has to rank epochs.
    """
    rng = np.random.default_rng(seed)
    by_school = defaultdict(list)
    for sid in student_ids:
        by_school[student_school_map[sid]].append(sid)

    train_ids, val_ids, test_ids = [], [], []
    n_singleton, n_pairs = 0, 0
    for students in by_school.values():
        students = list(students)
        rng.shuffle(students)
        n = len(students)
        if n == 1:
            train_ids.extend(students)
            n_singleton += 1
            continue
        if n == 2:
            train_ids.append(students[0])
            test_ids.append(students[1])
            n_pairs += 1
            continue
        n_train = min(max(int(round(n * train_frac)), 1), n - 2)
        n_val = min(max(int(round(n * val_frac)), 1), n - n_train - 1)
        train_ids.extend(students[:n_train])
        val_ids.extend(students[n_train:n_train + n_val])
        test_ids.extend(students[n_train + n_val:])

    if verbose:
        if n_singleton:
            print(f"Note: {n_singleton} school(s) have 1 student, placed entirely in train.")
        if n_pairs:
            print(f"Note: {n_pairs} school(s) have 2 students, split train/test only.")
        total = len(train_ids) + len(val_ids) + len(test_ids)
        print(f"split: {len(train_ids)} train / {len(val_ids)} val / {len(test_ids)} test "
              f"= {total} students across {len(by_school)} schools "
              f"({len(train_ids) / total:.1%} / {len(val_ids) / total:.1%} / "
              f"{len(test_ids) / total:.1%})")
    return train_ids, val_ids, test_ids


def subsample_by_school(student_ids, student_school_map, n_target, seed=0,
                        verbose=True) -> list:
    """Take whole schools at random until `n_target` students are reached.

    Whole schools rather than individual students: a random draw of a few hundred students
    from tens of thousands leaves most schools with one or two, and `stratified_school_split`
    sends 1- and 2-student schools entirely to train/test -- so val would be starved and a
    subsampled run would not exercise the same split behaviour as the full one.
    """
    by_school = defaultdict(list)
    for sid in student_ids:
        by_school[student_school_map[sid]].append(sid)

    schools = sorted(by_school)
    np.random.default_rng(seed).shuffle(schools)
    picked = []
    for school in schools:
        if len(picked) >= n_target:
            break
        picked.extend(by_school[school])
    n_schools = sum(1 for s in schools
                    if by_school[s] and by_school[s][0] in set(picked))
    if verbose:
        print(f"subsampled to {len(picked)} students across {n_schools} whole schools "
              f"(target {n_target}, from {len(student_ids)})")
    return sorted(picked)


def as_three_way(entry):
    """Normalize a split entry to (train, val, test). A two-way entry -- from a splits.csv
    written before this grew a validation set -- returns an empty val list, which every fit
    already treats as "no early stopping"."""
    entry = list(entry)
    if len(entry) == 2:
        return list(entry[0]), [], list(entry[1])
    if len(entry) == 3:
        return list(entry[0]), list(entry[1]), list(entry[2])
    raise ValueError(f"split entry must have 2 or 3 id lists, got {len(entry)}")


def generate_school_stratified_splits(student_ids, student_school_map, n_runs,
                                      train_frac=0.60, val_frac=0.20, base_seed=0,
                                      verbose=True) -> dict:
    """-> {run: (train_ids, val_ids, test_ids)}. Does not touch disk."""
    splits = {}
    for run in range(n_runs):
        if verbose:
            print(f"--- run {run} ---")
        splits[run] = stratified_school_split(student_ids, student_school_map,
                                              train_frac=train_frac, val_frac=val_frac,
                                              seed=base_seed + run, verbose=verbose)
    return splits








def filter_splits_to_available_students(splits, available_student_ids,
                                        verbose=True) -> dict:
    """Restrict every run to students present in a second dataset's population.

    Reports what each split loses, so a small overlap between the two workspaces does not
    go unnoticed.
    """
    available = set(available_student_ids)
    filtered = {}
    before = {name: 0 for name in SPLIT_NAMES}
    after = {name: 0 for name in SPLIT_NAMES}

    for run, entry in splits.items():
        parts = as_three_way(entry)
        kept = tuple([s for s in part if s in available] for part in parts)
        filtered[run] = kept
        for name, part, k in zip(SPLIT_NAMES, parts, kept):
            before[name] += len(part)
            after[name] += len(k)
        if verbose:
            desc = ", ".join(f"{name} {len(part)} -> {len(k)} "
                             f"({len(k) / max(len(part), 1):.1%} kept)"
                             for name, part, k in zip(SPLIT_NAMES, parts, kept))
            print(f"run {run}: {desc}")

    if verbose:
        desc = ", ".join(f"{name} {before[name]} -> {after[name]} "
                         f"({after[name] / max(before[name], 1):.1%} kept)"
                         for name in SPLIT_NAMES)
        print(f"\noverall: {desc}")
        for name in ("train", "test"):
            if before[name] and after[name] / before[name] < 0.5:
                print(f"WARNING: less than half of the {name} students are present in the "
                      f"second dataset -- results may not represent the intended population.")
    return filtered

## Padded batches

Per-student chronological sequences -> padded (T, B) tensors.

PC and DKT consume the same batches from the same index maps, so the two are always
compared on identical opportunities in identical order.

In [ ]:
def sequence_tuples(g, skill_idx, type_idx, problem_idx) -> list:
    """One student's rows -> (skill, correct, seconds, problem type, problem) tuples."""
    return list(zip(g["skill"].map(skill_idx), g["correct"].astype(int),
                    g["time_taken_seconds"].astype(float),
                    g["problem_type"].map(type_idx), g["problem_id"].map(problem_idx)))


def pad_sequences_pc(sequences):
    """-> (T, B) skill_idx, resp, ptype_idx, problem_idx, mask.

    `time_taken_seconds` is accepted for schema compatibility and discarded here; no model
    in these notebooks reads the time channel.
    """
    n_batch = len(sequences)
    t_max = max((len(s) for s in sequences), default=0)
    skill_idx = torch.zeros(t_max, n_batch, dtype=torch.long, device=device)
    resp = torch.zeros(t_max, n_batch, dtype=dtype, device=device)
    ptype_idx = torch.zeros(t_max, n_batch, dtype=torch.long, device=device)
    problem_idx = torch.zeros(t_max, n_batch, dtype=torch.long, device=device)
    mask = torch.zeros(t_max, n_batch, dtype=torch.bool, device=device)

    for b, seq in enumerate(sequences):
        for t, (k, x, _seconds, pt, pidx) in enumerate(seq):
            skill_idx[t, b] = k
            resp[t, b] = x
            ptype_idx[t, b] = pt
            problem_idx[t, b] = pidx
            mask[t, b] = True
    return skill_idx, resp, ptype_idx, problem_idx, mask


def build_padded_batch(skill_idx, type_idx, problem_idx, joint_df, student_ids):
    """-> (padded tensors, student ids present in batch order).

    Built once and reused across epochs, so a per-epoch validation score costs no groupby.
    """
    seq_map = {sid: sequence_tuples(g, skill_idx, type_idx, problem_idx)
               for sid, g in joint_df.groupby("student_id", sort=False)}
    ids_present = [s for s in student_ids if s in seq_map]
    return pad_sequences_pc([seq_map[s] for s in ids_present]), ids_present


def clone_state(module) -> dict:
    """Detached parameter snapshot, for rewinding to the best epoch."""
    return {k: v.detach().clone() for k, v in module.state_dict().items()}

## BKT baseline

BKT baseline: two-state HMM per skill, fit by vectorized Baum-Welch.

In [ ]:
@dataclass
class BKTParams:
    p_init: float
    p_learn: float
    p_slip: float
    p_guess: float

    def as_dict(self) -> dict:
        return dict(p_init=self.p_init, p_learn=self.p_learn,
                    p_slip=self.p_slip, p_guess=self.p_guess)


class BKTModel:
    """One skill's BKT. `fit` takes a list of per-student 0/1 sequences.

    Two departures from the textbook M-step, both defaults:

    `max_slip_plus_guess` caps p_slip + p_guess below 1. At or above 1 the mastered state
    stops predicting correctness better than the unmastered one and p_init / p_learn stop
    meaning mastery at all. Violations rescale both down proportionally, preserving their
    ratio, rather than clipping one.

    `*_prior_mean` / `*_prior_strength` put a Beta prior on slip and guess, applied as
    additive smoothing. When a skill is learned within the first opportunity or two almost
    no evidence remains in the unmastered state and the raw MLE for p_guess is
    unidentified; the prior shrinks it in exactly that case and washes out when evidence
    is abundant. Strength is in pseudo-observations; 0 recovers the raw MLE.
    """

    def __init__(self, n_iter: int = 100, tol: float = 1e-5,
                 init_params: Optional[BKTParams] = None, random_state: int = 0,
                 slip_bounds: tuple = (1e-4, 0.3), guess_bounds: tuple = (1e-4, 0.3),
                 max_slip_plus_guess: float = 0.9,
                 slip_prior_mean: float = 0.1, slip_prior_strength: float = 4.0,
                 guess_prior_mean: float = 0.1, guess_prior_strength: float = 4.0):
        self.n_iter = n_iter
        self.tol = tol
        self.random_state = random_state
        self.init_params = init_params
        self.slip_bounds = slip_bounds
        self.guess_bounds = guess_bounds
        assert 0 < max_slip_plus_guess < 1, "max_slip_plus_guess must be in (0, 1)"
        self.max_slip_plus_guess = max_slip_plus_guess
        self.slip_prior_mean = slip_prior_mean
        self.slip_prior_strength = slip_prior_strength
        self.guess_prior_mean = guess_prior_mean
        self.guess_prior_strength = guess_prior_strength
        self.params_: Optional[BKTParams] = None
        self.log_likelihood_history_ = []
        self.m_step_diagnostics_ = {}

    @staticmethod
    def _pad(sequences: list):
        n = len(sequences)
        max_len = max((len(s) for s in sequences), default=0)
        obs = np.zeros((n, max_len), dtype=np.int8)
        mask = np.zeros((n, max_len), dtype=bool)
        for i, s in enumerate(sequences):
            obs[i, :len(s)] = s
            mask[i, :len(s)] = True
        return obs, mask

    @staticmethod
    def _emission_prob(obs, slip, guess):
        """(N, T, 2): P(observed correctness | state 0 = not mastered, state 1 = mastered)."""
        p0 = np.where(obs == 1, guess, 1 - guess)
        p1 = np.where(obs == 1, 1 - slip, slip)
        return np.stack([p0, p1], axis=-1)

    def _e_step(self, obs, mask, params: BKTParams):
        n, t_max = obs.shape
        emis = np.clip(self._emission_prob(obs, params.p_slip, params.p_guess), EPS, 1.0)
        trans = np.array([[1 - params.p_learn, params.p_learn], [0.0, 1.0]])
        prior = np.array([1 - params.p_init, params.p_init])

        alpha = np.zeros((n, t_max, 2))
        c = np.ones((n, t_max))

        alpha0 = prior[None, :] * emis[:, 0, :]
        c0 = np.clip(alpha0.sum(axis=1), EPS, None)
        alpha[:, 0, :] = alpha0 / c0[:, None]
        c[:, 0] = np.where(mask[:, 0], c0, 1.0)

        # Padded steps carry alpha/beta forward unchanged rather than absorbing the fake
        # obs=0 as evidence. This matters most for the backward pass, which runs from the
        # end of the padded region straight through every padded position.
        for t in range(1, t_max):
            a_t = (alpha[:, t - 1, :] @ trans) * emis[:, t, :]
            c_t = np.clip(a_t.sum(axis=1), EPS, None)
            m = mask[:, t]
            alpha[:, t, :] = np.where(m[:, None], a_t / c_t[:, None], alpha[:, t - 1, :])
            c[:, t] = np.where(m, c_t, 1.0)

        beta = np.ones((n, t_max, 2))
        for t in range(t_max - 2, -1, -1):
            nxt = beta[:, t + 1, :] * emis[:, t + 1, :]
            beta_t = (nxt @ trans.T) / np.clip(c[:, t + 1], EPS, None)[:, None]
            m = mask[:, t + 1]
            beta[:, t, :] = np.where(m[:, None], beta_t, beta[:, t + 1, :])

        gamma = alpha * beta
        gamma = gamma / np.clip(gamma.sum(axis=-1, keepdims=True), EPS, None)

        xi = np.zeros((n, max(t_max - 1, 0), 2, 2))
        for t in range(t_max - 1):
            for i in range(2):
                for j in range(2):
                    xi[:, t, i, j] = (alpha[:, t, i] * trans[i, j] * emis[:, t + 1, j]
                                      * beta[:, t + 1, j]
                                      / np.clip(c[:, t + 1], EPS, None))

        gamma = gamma * mask[:, :, None]
        if t_max > 1:
            xi = xi * mask[:, 1:][:, :, None, None]
        log_lik = (np.log(np.clip(c, EPS, None)) * mask).sum()
        return gamma, xi, log_lik

    def _m_step(self, obs, mask, gamma, xi) -> BKTParams:
        p_init = gamma[:, 0, 1].sum() / max(obs.shape[0], 1)

        xi_sum = xi.sum(axis=(0, 1))
        p_learn = xi_sum[0, 1] / max(xi_sum[0, 0] + xi_sum[0, 1], EPS)

        mastered_w, not_mastered_w = gamma[:, :, 1], gamma[:, :, 0]
        slip_successes = (mastered_w * (obs == 0) * mask).sum()
        slip_trials = (mastered_w * mask).sum()
        guess_successes = (not_mastered_w * (obs == 1) * mask).sum()
        guess_trials = (not_mastered_w * mask).sum()

        raw_slip = slip_successes / max(slip_trials, EPS)
        raw_guess = guess_successes / max(guess_trials, EPS)
        p_slip = ((slip_successes + self.slip_prior_mean * self.slip_prior_strength)
                  / max(slip_trials + self.slip_prior_strength, EPS))
        p_guess = ((guess_successes + self.guess_prior_mean * self.guess_prior_strength)
                   / max(guess_trials + self.guess_prior_strength, EPS))
        map_slip, map_guess = p_slip, p_guess

        p_init = float(np.clip(p_init, 1e-4, 1 - 1e-4))
        p_learn = float(np.clip(p_learn, 1e-4, 1 - 1e-4))
        p_slip = float(np.clip(p_slip, *self.slip_bounds))
        p_guess = float(np.clip(p_guess, *self.guess_bounds))

        total = p_slip + p_guess
        if total > self.max_slip_plus_guess:
            scale = self.max_slip_plus_guess / total
            p_slip *= scale
            p_guess *= scale

        self.m_step_diagnostics_ = {
            "slip_trials": float(slip_trials), "guess_trials": float(guess_trials),
            "raw_mle_slip": float(raw_slip), "raw_mle_guess": float(raw_guess),
            "map_pre_bound_slip": float(map_slip), "map_pre_bound_guess": float(map_guess),
            "final_slip": p_slip, "final_guess": p_guess,
        }
        return BKTParams(p_init=p_init, p_learn=p_learn, p_slip=p_slip, p_guess=p_guess)

    def fit(self, sequences: list):
        sequences = [s for s in sequences if len(s) > 0]
        if not sequences:
            raise ValueError("No non-empty observation sequences to fit on.")
        obs, mask = self._pad(sequences)
        params = self.init_params or BKTParams(p_init=0.3, p_learn=0.15,
                                               p_slip=0.1, p_guess=0.25)

        self.log_likelihood_history_ = []
        prev_ll = -np.inf
        for _ in range(self.n_iter):
            gamma, xi, ll = self._e_step(obs, mask, params)
            self.log_likelihood_history_.append(ll)
            params = self._m_step(obs, mask, gamma, xi)
            if abs(ll - prev_ll) < self.tol:
                prev_ll = ll
                break
            prev_ll = ll

        self.params_ = params
        self._last_obs, self._last_mask = obs, mask
        return self

    def predict_mastery(self, sequences: Optional[list] = None) -> list:
        """Smoothed P(mastered) at every opportunity. Describes a trajectory after the
        fact; it sees the whole sequence, so never score held-out prediction with it."""
        if self.params_ is None:
            raise RuntimeError("Call fit() first.")
        obs, mask = (self._last_obs, self._last_mask) if sequences is None else self._pad(sequences)
        gamma, _, _ = self._e_step(obs, mask, self.params_)
        return [gamma[i, :int(mask[i].sum()), 1] for i in range(obs.shape[0])]

    def predict_correctness_probabilities(self, sequences: list) -> list:
        """Causal one-step-ahead P(correct): the prediction at step t uses only steps < t.

        This is the predictor every held-out evaluation in these notebooks uses.
        """
        if self.params_ is None:
            raise RuntimeError("Call fit() first.")
        obs, mask = self._pad(sequences)
        n, t_max = obs.shape
        p = self.params_
        trans = np.array([[1 - p.p_learn, p.p_learn], [0.0, 1.0]])
        emis = np.clip(self._emission_prob(obs, p.p_slip, p.p_guess), EPS, 1.0)

        belief = np.tile(np.array([1 - p.p_init, p.p_init]), (n, 1))
        preds = np.zeros((n, t_max))
        for t in range(t_max):
            preds[:, t] = belief[:, 0] * p.p_guess + belief[:, 1] * (1 - p.p_slip)
            updated = belief * emis[:, t, :]
            updated = updated / np.clip(updated.sum(axis=1, keepdims=True), EPS, None)
            belief = updated @ trans

        return [preds[i, :int(mask[i].sum())] for i in range(n)]

## DKT baseline

DKT baseline (Piech et al., 2015): an LSTM over one-hot (skill, correctness) tuples.

Fit on the same index maps, sequences and validation students as the PC, so the stopping
epoch is chosen the same way for both. An LSTM this size overfits sooner than the PC
filter, and letting only one model rewind to its best epoch would flatter whichever
happened to suit the fixed budget.

In [ ]:
class DKTModel(torch.nn.Module):
    def __init__(self, K, hidden_size=100, num_layers=1):
        super().__init__()
        self.K = K
        self.hidden_size = hidden_size
        self.lstm = torch.nn.LSTM(input_size=2 * K, hidden_size=hidden_size,
                                  num_layers=num_layers)
        self.output_layer = torch.nn.Linear(hidden_size, K)

    def forward(self, input_onehot):
        """(T, B, 2K) -> (T, B, K) predicted P(correct) for every skill.

        Output at t uses the hidden state BEFORE processing step t, so it is the causal
        prediction for that opportunity.
        """
        _, n_batch, _ = input_onehot.shape
        outputs, _ = self.lstm(input_onehot)
        h0 = torch.zeros(1, n_batch, self.hidden_size, dtype=input_onehot.dtype,
                         device=input_onehot.device)
        return torch.sigmoid(self.output_layer(torch.cat([h0, outputs[:-1]], dim=0)))


def encode_dkt_input(skill_idx, resp, K):
    """(T, B) -> (T, B, 2K) one-hot at skill_idx + K*correct."""
    return F.one_hot(skill_idx + K * resp.long(), num_classes=2 * K).to(dtype)


def dkt_loss(pred_seq, skill_idx, resp, mask):
    pred_at_skill = torch.gather(pred_seq, 2, skill_idx.unsqueeze(-1)).squeeze(-1)
    pred_at_skill = torch.clamp(pred_at_skill, 1e-6, 1 - 1e-6)
    resp_f = resp.to(dtype)
    bce = -(resp_f * torch.log(pred_at_skill) + (1 - resp_f) * torch.log(1 - pred_at_skill))
    return (bce * mask.to(dtype)).sum()


@torch.no_grad()
def dkt_val_logloss(model, padded, K) -> float:
    """`forward` is already causally shifted, so its masked BCE is the one-step-ahead
    log loss -- no separate recurrence needed, unlike the PC filter."""
    skill_idx_pad, resp_pad, _, _, mask_pad = padded
    n_obs = int(mask_pad.sum().item())
    if n_obs == 0:
        return float("nan")
    pred_seq = model(encode_dkt_input(skill_idx_pad, resp_pad, K))
    return float(dkt_loss(pred_seq, skill_idx_pad, resp_pad, mask_pad).item() / n_obs)


def fit_dkt_model(joint_df, train_ids, skill_idx, type_idx, problem_idx, hidden_size=100,
                  num_layers=1, n_epochs=20, batch_size=128, lr=0.01, seed=0,
                  max_grad_norm=5.0, val_ids=None, patience=None, verbose=True):
    """Index maps come from the PC fit, so both models see identical sequences."""
    K = len(skill_idx)
    seq_map = {sid: sequence_tuples(g, skill_idx, type_idx, problem_idx)
               for sid, g in joint_df.groupby("student_id", sort=False)}
    train_sequences = [seq_map[s] for s in train_ids if s in seq_map]

    model = DKTModel(K, hidden_size=hidden_size, num_layers=num_layers).to(device=device,
                                                                          dtype=dtype)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    rng = np.random.default_rng(seed)

    val_padded = None
    if val_ids is not None and len(val_ids) > 0:
        val_padded, val_ids_present = build_padded_batch(skill_idx, type_idx, problem_idx,
                                                         joint_df, val_ids)
        if verbose:
            print(f"validation: {len(val_ids_present)} students, "
                  f"{int(val_padded[4].sum().item())} opportunities")
    best = dict(epoch=-1, val_log_loss=float("inf"), state=None)
    val_history = []

    n = len(train_sequences)
    for epoch in range(n_epochs):
        order = rng.permutation(n)
        epoch_loss, n_obs = 0.0, 0
        for start in range(0, n, batch_size):
            batch_seqs = [train_sequences[i] for i in order[start:start + batch_size]]
            skill_idx_pad, resp_pad, _, _, mask_pad = pad_sequences_pc(batch_seqs)

            optimizer.zero_grad()
            pred_seq = model(encode_dkt_input(skill_idx_pad, resp_pad, K))
            loss = dkt_loss(pred_seq, skill_idx_pad, resp_pad, mask_pad)

            if not torch.isfinite(loss):
                if verbose:
                    print(f"  epoch {epoch:4d}: non-finite DKT loss, skipping this mini-batch")
                optimizer.zero_grad()
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            optimizer.step()

            epoch_loss += loss.item()
            n_obs += int(mask_pad.sum().item())
            if device.type == "mps":
                torch.mps.empty_cache()

        if val_padded is not None:
            vll = dkt_val_logloss(model, val_padded, K)
            improved = vll < best["val_log_loss"] - 1e-6
            val_history.append(dict(epoch=epoch, train_loss=epoch_loss, val_log_loss=vll))
            if improved:
                best = dict(epoch=epoch, val_log_loss=vll, state=clone_state(model))
            if verbose:
                print(f"epoch {epoch:4d}  train loss = {epoch_loss:.3f}  "
                      f"(avg {epoch_loss / max(n_obs, 1):.4f}/obs)  "
                      f"val log loss = {vll:.5f}{'  <- best' if improved else ''}")
            if patience is not None and epoch - best["epoch"] >= patience:
                if verbose:
                    print(f"early stop: {patience} epoch(s) with no validation improvement")
                break
        elif verbose and (epoch % 5 == 0 or epoch == n_epochs - 1):
            print(f"epoch {epoch:4d}  total loss = {epoch_loss:.3f} "
                  f"(avg {epoch_loss / max(n_obs, 1):.4f}/obs)")

    if best["state"] is not None:
        model.load_state_dict(best["state"])
        if verbose:
            print(f"restored epoch {best['epoch']} (best validation log loss "
                  f"{best['val_log_loss']:.5f})")

    return dict(model=model, skill_idx=skill_idx, type_idx=type_idx,
                problem_idx=problem_idx, K=K, hidden_size=hidden_size,
                val_history=pd.DataFrame(val_history), best_epoch=best["epoch"],
                best_val_log_loss=(best["val_log_loss"] if best["state"] is not None
                                   else float("nan")))


@torch.no_grad()
def causal_predictions_dkt_keyed(fit_dkt, joint_df, eval_ids) -> pd.DataFrame:
    """Predictions keyed by (student_id, skill, opportunity_index), so the three models'
    rows can be aligned exactly."""
    model = fit_dkt["model"]
    skill_idx, type_idx = fit_dkt["skill_idx"], fit_dkt["type_idx"]
    problem_idx, K = fit_dkt["problem_idx"], fit_dkt["K"]
    idx_to_skill = {v: k for k, v in skill_idx.items()}

    seq_map = {sid: sequence_tuples(g, skill_idx, type_idx, problem_idx)
               for sid, g in joint_df.groupby("student_id", sort=False)}
    eval_ids_present = [s for s in eval_ids if s in seq_map]
    skill_idx_pad, resp_pad, _, _, mask_pad = pad_sequences_pc(
        [seq_map[s] for s in eval_ids_present])

    pred_seq = model(encode_dkt_input(skill_idx_pad, resp_pad, K))
    pred_at_skill = torch.gather(pred_seq, 2, skill_idx_pad.unsqueeze(-1)).squeeze(-1)

    t_max, n_batch = skill_idx_pad.shape
    seen_count = np.zeros((n_batch, K), dtype=int)
    key_rows, y_true_list, y_pred_list = [], [], []
    for t in range(t_max):
        valid = mask_pad[t].cpu().numpy().astype(bool)
        skill_idx_t = skill_idx_pad[t].cpu().numpy()
        for b in np.nonzero(valid)[0]:
            k = int(skill_idx_t[b])
            key_rows.append((eval_ids_present[b], idx_to_skill[k], int(seen_count[b, k])))
            seen_count[b, k] += 1
        y_true_list.append(resp_pad[t].cpu().numpy()[valid])
        y_pred_list.append(pred_at_skill[t].cpu().numpy()[valid])

    df = pd.DataFrame(key_rows, columns=["student_id", "skill", "opportunity_index"])
    df["y_true"] = np.concatenate(y_true_list)
    df["y_pred"] = np.concatenate(y_pred_list)
    return df

## The probabilistic circuit

The region-graph probabilistic circuit.

A root sum node over R global archetypes, a product node partitioning skills into the
disjoint clusters declared in `curriculum`, a per-cluster sum node over L local states, and
per-skill Bernoulli leaves. Guess and slip are GLM-linked to skill, problem type (one type
held as the reference level) and a per-problem difficulty latent.

Fitting maximizes the exact log marginal likelihood -- `run_batch_pc_torch` returns it, and
autograd differentiates it. No EM, no ELBO.

`run_batch_pc_torch` (fitting) and `causal_predictions_pc_torch_padded` (evaluation) walk
the same recurrence; they are kept separate because one accumulates logZ and the other
emits one-step-ahead predictions before folding each response in.

In [ ]:
DIFFICULTY_SHRINKAGE_SIGMA = 2.0   # weak prior on raw difficulty
RELIABLE_TARGET_LOGIT = -3.48      # sigmoid(-3.48) ~= 0.03, i.e. "assume ~3% guess/slip"
RELIABLE_PRIOR_SIGMA = 0.5


def build_skill_index(skills, skill_cluster_map):
    skills = sorted(skills)
    clusters = sorted(set(skill_cluster_map.get(s, "unclustered") for s in skills))
    cluster_idx = {c: i for i, c in enumerate(clusters)}
    skill_idx = {s: i for i, s in enumerate(skills)}
    cluster_of = np.array([cluster_idx[skill_cluster_map.get(s, "unclustered")]
                           for s in skills])
    return skills, skill_idx, clusters, cluster_idx, cluster_of


def build_problem_index(joint_df: pd.DataFrame):
    """Difficulty is a per-problem latent, not per-skill: one problem generates several
    skill-opportunities and shares a single difficulty across all of them."""
    problem_ids = sorted(joint_df["problem_id"].unique())
    return problem_ids, {p: i for i, p in enumerate(problem_ids)}


class PCParamsBatched(torch.nn.Module):
    """Free parameters. `use_difficulty` / `use_forgetting` gate their term's *effect*, not
    its existence, so an ablation keeps the parameter count identical across conditions."""

    def __init__(self, K, C, R, L, n_types, n_problems, reliable_mask=None,
                 use_difficulty=True, use_forgetting=False):
        super().__init__()
        self.K, self.C, self.R, self.L, self.n_problems = K, C, R, L, n_problems
        self.use_difficulty = use_difficulty
        self.use_forgetting = use_forgetting

        if reliable_mask is None:
            reliable_mask = torch.zeros(K, dtype=dtype)
        self.register_buffer("reliable_mask", torch.as_tensor(reliable_mask, dtype=dtype))

        self.skill_guess_logit = torch.nn.Parameter(torch.full((K,), -1.0, dtype=dtype))
        self.skill_slip_logit = torch.nn.Parameter(torch.full((K,), -2.0, dtype=dtype))
        self.type_guess_offset_free = torch.nn.Parameter(torch.zeros(n_types - 1, dtype=dtype))
        self.type_slip_offset_free = torch.nn.Parameter(torch.zeros(n_types - 1, dtype=dtype))
        self.learn_rate_raw = torch.nn.Parameter(torch.full((K,), -1.7, dtype=dtype))

        self.root_prior_logit = torch.nn.Parameter(torch.zeros(R, dtype=dtype))
        self.cluster_weight_high_logit = torch.nn.Parameter(0.1 * torch.randn(R, C, dtype=dtype))
        self.leaf_param_logit = torch.nn.Parameter(torch.zeros(K, L, dtype=dtype))
        with torch.no_grad():
            if L == 2:
                self.leaf_param_logit[:, 0] = -0.85
                self.leaf_param_logit[:, 1] = 0.85

        self.kappa_guess = torch.nn.Parameter(torch.tensor(0.1, dtype=dtype))
        self.kappa_slip = torch.nn.Parameter(torch.tensor(0.1, dtype=dtype))
        self.difficulty_raw = torch.nn.Parameter(torch.zeros(n_problems, dtype=dtype))
        self.forget_rate_raw = torch.nn.Parameter(torch.full((K,), -3.0, dtype=dtype))

    def type_guess_offset_vec(self):
        return torch.cat([torch.zeros(1, dtype=dtype, device=self.type_guess_offset_free.device),
                          self.type_guess_offset_free])

    def type_slip_offset_vec(self):
        return torch.cat([torch.zeros(1, dtype=dtype, device=self.type_slip_offset_free.device),
                          self.type_slip_offset_free])

    def learn_rate_vec(self):
        return torch.sigmoid(self.learn_rate_raw)

    def forget_rate_vec(self):
        """Exact zeros when disabled, so the predict step reduces exactly to the classical
        absorbing-mastery transition rather than an approximation of it."""
        if not self.use_forgetting:
            return torch.zeros(self.K, dtype=dtype, device=self.forget_rate_raw.device)
        return torch.sigmoid(self.forget_rate_raw)

    def root_prior(self):
        return torch.softmax(self.root_prior_logit, dim=0)

    def cluster_weight_high(self):
        return torch.sigmoid(self.cluster_weight_high_logit)

    def leaf_param(self):
        return torch.sigmoid(self.leaf_param_logit)

    def difficulty_vec(self):
        """Exact zeros when disabled, so guess/slip reduce exactly to the skill+type GLM."""
        if not self.use_difficulty:
            return torch.zeros(self.n_problems, dtype=dtype, device=self.difficulty_raw.device)
        d = self.difficulty_raw - self.difficulty_raw.mean()
        return d / (self.difficulty_raw.std() + 1e-8)

    def difficulty_prior_penalty(self, sigma=DIFFICULTY_SHRINKAGE_SIGMA):
        if not self.use_difficulty:
            return torch.zeros((), dtype=dtype, device=self.difficulty_raw.device)
        return 0.5 * torch.sum((self.difficulty_raw / sigma) ** 2)

    def reliable_prior_penalty(self, target_logit=RELIABLE_TARGET_LOGIT,
                               sigma=RELIABLE_PRIOR_SIGMA):
        """Soft Gaussian pull toward near-zero-noise guess/slip for skills marked reliable.
        Zero contribution for the default mask."""
        g = 0.5 * torch.sum(self.reliable_mask * ((self.skill_guess_logit - target_logit) / sigma) ** 2)
        s = 0.5 * torch.sum(self.reliable_mask * ((self.skill_slip_logit - target_logit) / sigma) ** 2)
        return g + s

    @torch.no_grad()
    def clamp_params_(self):
        self.skill_guess_logit.clamp_(-6.0, 6.0)
        self.skill_slip_logit.clamp_(-6.0, 6.0)
        self.type_guess_offset_free.clamp_(-4.0, 4.0)
        self.type_slip_offset_free.clamp_(-4.0, 4.0)
        self.learn_rate_raw.clamp_(-6.0, 6.0)
        self.forget_rate_raw.clamp_(-6.0, 6.0)
        self.root_prior_logit.clamp_(-6.0, 6.0)
        self.cluster_weight_high_logit.clamp_(-6.0, 6.0)
        self.leaf_param_logit.clamp_(-6.0, 6.0)
        self.kappa_guess.clamp_(-3.0, 3.0)
        self.kappa_slip.clamp_(-3.0, 3.0)
        self.difficulty_raw.clamp_(-5.0, 5.0)

    def param_summary(self):
        return [(name, float(p.min()), float(p.max())) for name, p in self.named_parameters()]


def _init_state(params, n_batch, C, R):
    """(W, Q, theta): archetype belief, per-cluster local-state belief, per-skill mastery."""
    W = params.root_prior().unsqueeze(0).repeat(n_batch, 1).clone()
    cluster_weight_high_t = params.cluster_weight_high()
    Q = torch.zeros(n_batch, C, R, params.L, dtype=dtype, device=device)
    for c in range(C):
        Q[:, c, :, 1] = cluster_weight_high_t[:, c]
        Q[:, c, :, 0] = 1 - cluster_weight_high_t[:, c]
    theta = params.leaf_param().unsqueeze(0).unsqueeze(2).repeat(n_batch, 1, R, 1).clone()
    return W, Q, theta


def run_batch_pc_torch(params, cluster_of_t, R, L, C, K, skill_idx, resp, ptype_idx,
                       problem_idx, mask, normalize_per_student=False):
    """Exact per-step log marginal likelihood, summed over the batch.

    `normalize_per_student=False` sums every valid (student, opportunity) term, so longer
    sequences weigh proportionally more. True averages within a student first, so every
    student contributes equally -- this changes the magnitude of the returned value
    substantially, and the prior penalties added by the caller do not rescale with it.
    """
    t_max, n_batch = skill_idx.shape

    W, Q, theta = _init_state(params, n_batch, C, R)
    difficulty_t = params.difficulty_vec()
    forget_rate_vec = params.forget_rate_vec()
    type_g_off_vec = params.type_guess_offset_vec()
    type_s_off_vec = params.type_slip_offset_vec()
    lr_vec = params.learn_rate_vec()

    student_total_logZ = torch.zeros(n_batch, dtype=dtype, device=device)
    student_n_opportunities = torch.zeros(n_batch, dtype=dtype, device=device)

    for t in range(t_max):
        skill_idx_t, resp_t, mask_t = skill_idx[t], resp[t], mask[t]
        skill_onehot = F.one_hot(skill_idx_t, K).to(dtype)
        cluster_onehot = F.one_hot(cluster_of_t[skill_idx_t], C).to(dtype)
        m4 = mask_t[:, None, None, None]

        lr_t = lr_vec[skill_idx_t]
        forget_t = forget_rate_vec[skill_idx_t]
        d_t = difficulty_t[problem_idx[t]]

        # predict: learn/forget on the tested skill only
        theta_tested = (theta * skill_onehot[:, :, None, None]).sum(dim=1)
        theta_tested_new = (theta_tested * (1 - forget_t[:, None, None])
                            + (1 - theta_tested) * lr_t[:, None, None])
        theta = torch.where(m4, (theta * (1 - skill_onehot)[:, :, None, None]
                                 + theta_tested_new[:, None, :, :] * skill_onehot[:, :, None, None]),
                            theta)

        guess = torch.sigmoid(params.skill_guess_logit[skill_idx_t]
                              + type_g_off_vec[ptype_idx[t]] - params.kappa_guess * d_t)
        slip = torch.sigmoid(params.skill_slip_logit[skill_idx_t]
                             + type_s_off_vec[ptype_idx[t]] + params.kappa_slip * d_t)

        theta_tested = (theta * skill_onehot[:, :, None, None]).sum(dim=1)
        Q_tested = (Q * cluster_onehot[:, :, None, None]).sum(dim=1)

        p_correct = (theta_tested * (1 - slip)[:, None, None]
                     + (1 - theta_tested) * guess[:, None, None])
        lik = torch.where(resp_t[:, None, None] == 1, p_correct, 1 - p_correct)
        lik_c_r = (Q_tested * lik).sum(dim=2)
        log_lik_r = torch.log(torch.clamp(lik_c_r, min=1e-12))

        # correct: local state, then mastery, then the archetype weights
        new_Q_tested = Q_tested * lik
        new_Q_tested = new_Q_tested / torch.clamp(new_Q_tested.sum(dim=2, keepdim=True), min=1e-12)
        Q = torch.where(m4, (Q * (1 - cluster_onehot)[:, :, None, None]
                             + new_Q_tested[:, None, :, :] * cluster_onehot[:, :, None, None]), Q)

        p1 = torch.where(resp_t == 1, 1 - slip, slip)
        p0 = torch.where(resp_t == 1, guess, 1 - guess)
        num = theta_tested * p1[:, None, None]
        den = num + (1 - theta_tested) * p0[:, None, None]
        new_theta_tested = num / torch.clamp(den, min=1e-12)
        theta = torch.where(m4, (theta * (1 - skill_onehot)[:, :, None, None]
                                 + new_theta_tested[:, None, :, :] * skill_onehot[:, :, None, None]),
                            theta)

        unnorm_log_W = torch.log(torch.clamp(W, min=1e-12)) + log_lik_r
        mmax = unnorm_log_W.max(dim=1, keepdim=True).values
        logZ_per_student = mmax.squeeze(1) + torch.log(torch.sum(torch.exp(unnorm_log_W - mmax), dim=1))
        W_new = torch.exp(unnorm_log_W - mmax)
        W_new = W_new / W_new.sum(dim=1, keepdim=True)
        W = torch.where(mask_t[:, None], W_new, W)

        student_total_logZ = student_total_logZ + logZ_per_student * mask_t.to(dtype)
        student_n_opportunities = student_n_opportunities + mask_t.to(dtype)

    if normalize_per_student:
        return (student_total_logZ / torch.clamp(student_n_opportunities, min=1.0)).sum()
    return student_total_logZ.sum()


@torch.no_grad()
def filter_recurrence(fit, padded, allowed_lookup=None, key_ids=None, W_init=None):
    """The evaluation-side pass of the same recurrence `run_batch_pc_torch` fits with.

    At each step the prediction is taken from the belief state BEFORE that step's response
    is folded in, then local state, mastery and the archetype weights are updated.

    `allowed_lookup` is a length-K bool tensor; opportunities on skills outside it are
    treated as padding for update purposes, so belief reflects only the allowed evidence.
    `key_ids` (student ids in batch order) turns on (student_id, skill, opportunity_index)
    key tracking, built in this same loop so a key names the prediction it is attached to
    by construction.

    `W_init` is an (n_batch, R) array of starting archetype beliefs, replacing the
    population root prior -- this is what carries a belief transferred from another fit.
    The full predict/update recurrence still runs afterward, so a student's own evidence
    accumulates on top of the transferred prior rather than being displaced by it.

    -> (y_true, y_pred, key_rows, W_final).
    """
    skill_idx_pad, resp_pad, ptype_idx_pad, problem_idx_pad, mask_pad = padded
    params = fit["params"]
    R, L, C, K = fit["R"], fit["L"], fit["C"], fit["K"]
    cluster_of_t = fit["cluster_of_t"]
    t_max, n_batch = skill_idx_pad.shape

    W, Q, theta = _init_state(params, n_batch, C, R)
    if W_init is not None:
        W = torch.as_tensor(W_init, dtype=dtype, device=device)
    difficulty_t = params.difficulty_vec()
    forget_rate_vec = params.forget_rate_vec()
    type_g_off_vec = params.type_guess_offset_vec()
    type_s_off_vec = params.type_slip_offset_vec()
    lr_vec = params.learn_rate_vec()

    seen_count = np.zeros((n_batch, K), dtype=int) if key_ids is not None else None
    key_rows, y_true_list, y_pred_list = [], [], []

    for t in range(t_max):
        skill_idx_t, resp_t, mask_t = skill_idx_pad[t], resp_pad[t], mask_pad[t]
        update_mask_t = mask_t if allowed_lookup is None else mask_t & allowed_lookup[skill_idx_t]
        skill_onehot = F.one_hot(skill_idx_t, K).to(dtype)
        cluster_onehot = F.one_hot(cluster_of_t[skill_idx_t], C).to(dtype)
        m4 = update_mask_t[:, None, None, None]

        lr_t = lr_vec[skill_idx_t]
        forget_t = forget_rate_vec[skill_idx_t]
        d_t = difficulty_t[problem_idx_pad[t]]

        theta_tested = (theta * skill_onehot[:, :, None, None]).sum(dim=1)
        theta_tested_new = (theta_tested * (1 - forget_t[:, None, None])
                            + (1 - theta_tested) * lr_t[:, None, None])
        theta = torch.where(m4, (theta * (1 - skill_onehot)[:, :, None, None]
                                 + theta_tested_new[:, None, :, :] * skill_onehot[:, :, None, None]),
                            theta)

        guess = torch.sigmoid(params.skill_guess_logit[skill_idx_t]
                              + type_g_off_vec[ptype_idx_pad[t]] - params.kappa_guess * d_t)
        slip = torch.sigmoid(params.skill_slip_logit[skill_idx_t]
                             + type_s_off_vec[ptype_idx_pad[t]] + params.kappa_slip * d_t)

        theta_tested = (theta * skill_onehot[:, :, None, None]).sum(dim=1)
        Q_tested = (Q * cluster_onehot[:, :, None, None]).sum(dim=1)
        p_correct_rl = (theta_tested * (1 - slip)[:, None, None]
                        + (1 - theta_tested) * guess[:, None, None])

        pred = (W * (Q_tested * p_correct_rl).sum(dim=2)).sum(dim=1)
        valid = mask_t.cpu().numpy().astype(bool)
        if key_ids is not None:
            skills_sorted = fit["skills_sorted"]
            skill_idx_t_np = skill_idx_t.cpu().numpy()
            for b in np.nonzero(valid)[0]:
                k = int(skill_idx_t_np[b])
                key_rows.append((key_ids[b], skills_sorted[k], int(seen_count[b, k])))
                seen_count[b, k] += 1
        y_true_list.append(resp_t.cpu().numpy()[valid])
        y_pred_list.append(pred.cpu().numpy()[valid])

        lik = torch.where(resp_t[:, None, None] == 1, p_correct_rl, 1 - p_correct_rl)
        new_Q_tested = Q_tested * lik
        new_Q_tested = new_Q_tested / torch.clamp(new_Q_tested.sum(dim=2, keepdim=True), min=1e-12)
        Q = torch.where(m4, (Q * (1 - cluster_onehot)[:, :, None, None]
                             + new_Q_tested[:, None, :, :] * cluster_onehot[:, :, None, None]), Q)

        p1 = torch.where(resp_t == 1, 1 - slip, slip)
        p0 = torch.where(resp_t == 1, guess, 1 - guess)
        num = theta_tested * p1[:, None, None]
        den = num + (1 - theta_tested) * p0[:, None, None]
        new_theta = num / torch.clamp(den, min=1e-12)
        theta = torch.where(m4, (theta * (1 - skill_onehot)[:, :, None, None]
                                 + new_theta[:, None, :, :] * skill_onehot[:, :, None, None]),
                            theta)

        lik_c_r = (Q_tested * lik).sum(dim=2)
        unnorm_log_W = (torch.log(torch.clamp(W, min=1e-12))
                        + torch.log(torch.clamp(lik_c_r, min=1e-12)))
        mmax = unnorm_log_W.max(dim=1, keepdim=True).values
        W_new = torch.exp(unnorm_log_W - mmax)
        W_new = W_new / W_new.sum(dim=1, keepdim=True)
        W = torch.where(update_mask_t[:, None], W_new, W)

    return (np.concatenate(y_true_list), np.concatenate(y_pred_list), key_rows, W)


def _keyed_frame(y_true, y_pred, key_rows) -> pd.DataFrame:
    df = pd.DataFrame(key_rows, columns=["student_id", "skill", "opportunity_index"])
    df["y_true"] = y_true
    df["y_pred"] = y_pred
    return df


@torch.no_grad()
def causal_predictions_pc_torch_padded(fit, padded):
    """One-step-ahead (y_true, y_pred) over an already-padded batch. `fit` is read for
    params/R/L/C/K/cluster_of_t only, so a partial view from inside the training loop works
    as well as a finished fit."""
    y_true, y_pred, _, _ = filter_recurrence(fit, padded)
    return y_true, y_pred




@torch.no_grad()
def causal_predictions_pc_torch_keyed(fit, joint_df, eval_ids) -> pd.DataFrame:
    """Predictions keyed by (student_id, skill, opportunity_index), for aligning models."""
    padded, ids_present = build_padded_batch(fit["skill_idx"], fit["type_idx"],
                                             fit["problem_idx"], joint_df, eval_ids)
    y_true, y_pred, key_rows, _ = filter_recurrence(fit, padded, key_ids=ids_present)
    return _keyed_frame(y_true, y_pred, key_rows)


def allowed_skill_lookup(fit, allowed_skills):
    """Length-K bool tensor marking the skills belief is allowed to update on."""
    lookup = torch.zeros(fit["K"], dtype=torch.bool, device=device)
    if allowed_skills is None:
        return None
    for s in allowed_skills:
        if s in fit["skill_idx"]:
            lookup[fit["skill_idx"][s]] = True
    return lookup


@torch.no_grad()
def compute_final_archetype_belief(fit, joint_df, student_ids, allowed_skills=None) -> dict:
    """student -> final archetype posterior W, after all their allowed-skill evidence.

    Restricting to a skill subset (e.g. the skills two workspaces share) is what makes the
    signal comparable across two independent fits.
    """
    padded, ids_present = build_padded_batch(fit["skill_idx"], fit["type_idx"],
                                             fit["problem_idx"], joint_df, student_ids)
    _, _, _, W = filter_recurrence(fit, padded,
                                   allowed_lookup=allowed_skill_lookup(fit, allowed_skills))
    return {sid: W[i].cpu().numpy() for i, sid in enumerate(ids_present)}


@torch.no_grad()
def pc_val_logloss(fit_view, padded) -> float:
    """Mean causal log loss over a held-out batch -- the quantity the training loop watches,
    defined identically to the log_loss the comparison goes on to report."""
    y_true, y_pred = causal_predictions_pc_torch_padded(fit_view, padded)
    if len(y_true) == 0:
        return float("nan")
    p = np.clip(y_pred, EPS, 1 - EPS)
    return float(-np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)))


def fit_pc_model_torch(joint_df, skill_cluster_map, train_ids, R=2, L=2, n_epochs=20,
                       batch_size=128, lr=0.01, seed=0, max_grad_norm=5.0,
                       reliable_skills=None, use_difficulty=True, use_forgetting=False,
                       normalize_per_student=False, val_ids=None, patience=None,
                       verbose=True):
    """Fit by Adam on the exact log-likelihood.

    `val_ids` students choose the stopping epoch and contribute no gradient; the fit rewinds
    to the best-validating epoch at the end. `patience` stops early after that many epochs
    without improvement; None spends the full budget and still rewinds.
    """
    skills_sorted, skill_idx, clusters, _, cluster_of = build_skill_index(
        joint_df["skill"].unique(), skill_cluster_map)
    K, C = len(skills_sorted), len(clusters)
    types = sorted(joint_df["problem_type"].unique())
    type_idx = {t: i for i, t in enumerate(types)}
    problem_ids, problem_idx = build_problem_index(joint_df)

    seq_map = {sid: sequence_tuples(g, skill_idx, type_idx, problem_idx)
               for sid, g in joint_df.groupby("student_id", sort=False)}
    train_sequences = [seq_map[s] for s in train_ids if s in seq_map]

    cluster_of_t = torch.tensor(cluster_of, dtype=torch.long, device=device)
    reliable_mask = np.array([1.0 if s in (reliable_skills or set()) else 0.0
                              for s in skills_sorted])
    params = PCParamsBatched(K, C, R, L, len(types), len(problem_ids),
                             reliable_mask=reliable_mask, use_difficulty=use_difficulty,
                             use_forgetting=use_forgetting).to(device)
    optimizer = torch.optim.Adam(params.parameters(), lr=lr)
    rng = np.random.default_rng(seed)

    fit_view = dict(params=params, R=R, L=L, C=C, K=K, cluster_of_t=cluster_of_t)
    val_padded = None
    if val_ids is not None and len(val_ids) > 0:
        val_padded, val_ids_present = build_padded_batch(skill_idx, type_idx, problem_idx,
                                                         joint_df, val_ids)
        if verbose:
            print(f"validation: {len(val_ids_present)} students, "
                  f"{int(val_padded[4].sum().item())} opportunities")
    best = dict(epoch=-1, val_log_loss=float("inf"), state=None)
    val_history = []

    n = len(train_sequences)
    for epoch in range(n_epochs):
        order = rng.permutation(n)
        epoch_loss, n_obs = 0.0, 0
        for start in range(0, n, batch_size):
            batch_seqs = [train_sequences[i] for i in order[start:start + batch_size]]
            padded = pad_sequences_pc(batch_seqs)

            optimizer.zero_grad()
            total_logZ = run_batch_pc_torch(params, cluster_of_t, R, L, C, K, *padded,
                                            normalize_per_student=normalize_per_student)
            loss = (-total_logZ + params.difficulty_prior_penalty()
                    + params.reliable_prior_penalty())

            if not torch.isfinite(loss):
                if verbose:
                    print(f"  epoch {epoch:4d}: non-finite loss, skipping this mini-batch")
                optimizer.zero_grad()
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(params.parameters(), max_norm=max_grad_norm)
            optimizer.step()
            params.clamp_params_()

            epoch_loss += loss.item()
            n_obs += int(padded[4].sum().item())
            if device.type == "mps":
                torch.mps.empty_cache()

        if val_padded is not None:
            vll = pc_val_logloss(fit_view, val_padded)
            improved = vll < best["val_log_loss"] - 1e-6
            val_history.append(dict(epoch=epoch, train_loss=epoch_loss, val_log_loss=vll))
            if improved:
                best = dict(epoch=epoch, val_log_loss=vll, state=clone_state(params))
            if verbose:
                print(f"epoch {epoch:4d}  train loss = {epoch_loss:.3f}  "
                      f"(avg {epoch_loss / max(n_obs, 1):.4f}/obs)  "
                      f"val log loss = {vll:.5f}{'  <- best' if improved else ''}")
            if patience is not None and epoch - best["epoch"] >= patience:
                if verbose:
                    print(f"early stop: {patience} epoch(s) with no validation improvement")
                break
        elif verbose and (epoch % 10 == 0 or epoch == n_epochs - 1):
            print(f"epoch {epoch:4d}  total loss = {epoch_loss:.3f} "
                  f"(avg {epoch_loss / max(n_obs, 1):.4f}/obs)")

    if best["state"] is not None:
        params.load_state_dict(best["state"])
        if verbose:
            print(f"restored epoch {best['epoch']} (best validation log loss "
                  f"{best['val_log_loss']:.5f})")

    return dict(params=params, skill_idx=skill_idx, skills_sorted=skills_sorted,
                cluster_of=cluster_of, clusters=clusters, types=types, type_idx=type_idx,
                n_types=len(types), R=R, L=L, C=C, cluster_of_t=cluster_of_t, K=K,
                problem_ids=problem_ids, problem_idx=problem_idx,
                n_problems=len(problem_ids), use_difficulty=use_difficulty,
                use_forgetting=use_forgetting, normalize_per_student=normalize_per_student,
                val_history=pd.DataFrame(val_history), best_epoch=best["epoch"],
                best_val_log_loss=(best["val_log_loss"] if best["state"] is not None
                                   else float("nan")))

## PC vs BKT vs DKT

PC vs BKT vs DKT on identical held-out opportunities.

Every model emits predictions keyed by (student_id, skill, opportunity_index); the keys are
inner-joined, so each reported number is computed over exactly the same rows for all three.

In [ ]:
MODEL_NAMES = ["pc", "bkt", "dkt"]


def bkt_causal_predictions_keyed(joint_df, train_ids, test_ids, bkt_kwargs=None) -> pd.DataFrame:
    """One BKT per skill, fit on the training students, scored causally on the held-out ones."""
    rows = []
    for skill, g in joint_df.groupby("skill"):
        seq_map = {sid: sub["correct"].to_numpy(dtype=np.int8)
                   for sid, sub in g.groupby("student_id")}
        train_seq = [seq_map[s] for s in train_ids if s in seq_map]
        test_seq = {s: seq_map[s] for s in test_ids if s in seq_map}
        if not train_seq or not test_seq:
            continue
        model = BKTModel(**(bkt_kwargs or {})).fit(train_seq)
        sids, seqs = list(test_seq.keys()), list(test_seq.values())
        for sid, true_arr, pred_arr in zip(sids, seqs,
                                           model.predict_correctness_probabilities(seqs)):
            for opp_idx, (yt, yp) in enumerate(zip(true_arr, pred_arr)):
                rows.append((sid, skill, opp_idx, int(yt), float(yp)))
    return pd.DataFrame(rows, columns=["student_id", "skill", "opportunity_index",
                                       "y_true", "y_pred"])


def align_multi_model_predictions(model_dfs: dict, warn: bool = True) -> pd.DataFrame:
    """Inner-join every model's keyed predictions, asserting they agree on ground truth."""
    names = list(model_dfs)
    merged = model_dfs[names[0]].rename(columns={"y_pred": f"y_pred_{names[0]}"})
    for name in names[1:]:
        other = model_dfs[name][["student_id", "skill", "opportunity_index",
                                 "y_true", "y_pred"]].rename(
            columns={"y_pred": f"y_pred_{name}", "y_true": f"y_true_{name}"})
        merged = merged.merge(other, on=["student_id", "skill", "opportunity_index"])
        assert not (merged["y_true"] != merged[f"y_true_{name}"]).any(), \
            f"ground truth mismatch between the first model and {name!r}"
        merged = merged.drop(columns=[f"y_true_{name}"])

    if warn:
        keys = {k for df in model_dfs.values()
                for k in zip(df["student_id"], df["skill"], df["opportunity_index"])}
        dropped = keys - set(zip(merged["student_id"], merged["skill"],
                                 merged["opportunity_index"]))
        if dropped:
            print(f"WARNING: {len(dropped)} triple(s) dropped from the aligned comparison "
                  f"across all {len(names)} models.")
    return merged


def _metric_row(sub, model_names, base: dict) -> dict:
    row = dict(base, n=len(sub))
    for name in model_names:
        m = compute_binary_classification_metrics(sub["y_true"].to_numpy(float),
                                                  sub[f"y_pred_{name}"].to_numpy())
        row[f"auc_{name}"] = m["auc"]
        row[f"log_loss_{name}"] = m["log_loss"]
    return row


def cold_warm_breakdown_multi(merged_df, model_names, threshold=3) -> pd.DataFrame:
    """Where joint modeling should help: a student's first exposures to a skill, where BKT
    has only its flat prior while the PC can borrow evidence from correlated skills."""
    rows = []
    for label, sub in [(f"cold (< {threshold})",
                        merged_df[merged_df["opportunity_index"] < threshold]),
                       (f"warm (>= {threshold})",
                        merged_df[merged_df["opportunity_index"] >= threshold])]:
        if not sub.empty:
            rows.append(_metric_row(sub, model_names, dict(regime=label)))
    return pd.DataFrame(rows)


def per_skill_breakdown_multi(merged_df, model_names) -> pd.DataFrame:
    return pd.DataFrame([_metric_row(sub, model_names, dict(skill=skill))
                         for skill, sub in merged_df.groupby("skill")])


def per_skill_cold_start_breakdown_multi(merged_df, model_names, threshold=3) -> pd.DataFrame:
    """The skill x regime cross-tabulation the two pooled breakdowns each collapse away.
    Regime labels are the bare strings the heatmap pivots on."""
    rows = []
    for regime, sub_regime in [("cold", merged_df[merged_df["opportunity_index"] < threshold]),
                               ("warm", merged_df[merged_df["opportunity_index"] >= threshold])]:
        if sub_regime.empty:
            continue
        for skill, sub in sub_regime.groupby("skill"):
            rows.append(_metric_row(sub, model_names, dict(skill=skill, regime=regime)))
    return pd.DataFrame(rows)


def rank_correlation_multi(merged_df, model_names) -> pd.DataFrame:
    """Pooled AUC and log-loss can disagree -- AUC reads only rank order, log-loss is a
    proper scoring rule. This says how far apart the models' rankings actually are."""
    rows = []
    for i, name_a in enumerate(model_names):
        for name_b in model_names[i + 1:]:
            rho, pval = spearmanr(merged_df[f"y_pred_{name_a}"], merged_df[f"y_pred_{name_b}"])
            rows.append(dict(model_a=name_a, model_b=name_b, spearman_rho=rho,
                             p_value=pval, n=len(merged_df)))
    return pd.DataFrame(rows)


def pooled_vs_weighted_multi(merged_df, per_skill_df, model_names) -> pd.DataFrame:
    """Pooled metrics beside the skill-count-weighted average of the per-skill metrics."""
    rows = []
    for key in ("auc", "log_loss"):
        row = dict(metric=key)
        for name in model_names:
            pooled = compute_binary_classification_metrics(
                merged_df["y_true"].to_numpy(float), merged_df[f"y_pred_{name}"].to_numpy())
            row[f"{name}_pooled"] = pooled[key]
            row[f"{name}_weighted_avg"] = np.average(per_skill_df[f"{key}_{name}"],
                                                     weights=per_skill_df["n"])
        rows.append(row)
    return pd.DataFrame(rows)


def full_diagnostic_comparison_torch_multi(joint_df, skill_cluster_map, train_ids, test_ids,
                                           fit_pc, fit_dkt, bkt_kwargs=None,
                                           cold_threshold=3) -> dict:
    """`fit_pc` and `fit_dkt` must already be fit on `train_ids`; BKT is fit here."""
    merged = align_multi_model_predictions({
        "pc": causal_predictions_pc_torch_keyed(fit_pc, joint_df, test_ids),
        "bkt": bkt_causal_predictions_keyed(joint_df, train_ids, test_ids,
                                            bkt_kwargs=bkt_kwargs),
        "dkt": causal_predictions_dkt_keyed(fit_dkt, joint_df, test_ids),
    })
    return dict(
        merged=merged,
        cold_warm=cold_warm_breakdown_multi(merged, MODEL_NAMES, threshold=cold_threshold),
        per_skill=per_skill_breakdown_multi(merged, MODEL_NAMES),
        per_skill_cold_start=per_skill_cold_start_breakdown_multi(
            merged, MODEL_NAMES, threshold=cold_threshold),
        rank_corr=rank_correlation_multi(merged, MODEL_NAMES),
    )


def multi_run_school_stratified_evaluation_three_way(
        joint_df, skill_cluster_map, splits, R=2, L=2, pc_n_epochs=20, dkt_n_epochs=20,
        dkt_hidden_size=100, batch_size=128, lr=0.01, reliable_skills=None,
        cold_threshold=3, bkt_kwargs=None, seed=0, verbose_filter=True,
        eval_splits=("val", "test"), patience=None, verbose=True):
    """Fit and score all three models on every split replicate. -> (summary, raw_runs).

    Every table carries an `eval_split` column: "test" is the headline, "val" is reported so
    it is visible whether the two agree. A large gap means the epoch choice bought more on
    val than it generalizes to.

    Splits are first filtered to students present in `joint_df`, so a splits.csv generated
    on the other workspace can be reused directly. That is a no-op when the populations
    already match.
    """
    splits = filter_splits_to_available_students(splits, joint_df["student_id"].unique(),
                                                 verbose=verbose_filter)
    pooled_rows, cold_warm_rows, per_skill_rows = [], [], []
    rank_corr_rows, per_skill_cs_rows, fit_info_rows = [], [], []

    for run, entry in splits.items():
        train_ids, val_ids, test_ids = as_three_way(entry)
        if len(train_ids) == 0 or len(test_ids) == 0:
            print(f"--- run {run}: SKIPPED (train={len(train_ids)}, test={len(test_ids)} "
                  f"after filtering -- not enough overlap to fit/evaluate) ---")
            continue

        if verbose:
            print(f"--- run {run}: {len(train_ids)} train / {len(val_ids)} val / "
                  f"{len(test_ids)} test students ---")

        fit_pc = fit_pc_model_torch(joint_df, skill_cluster_map, train_ids, R=R, L=L,
                                    n_epochs=pc_n_epochs, batch_size=batch_size, lr=lr,
                                    reliable_skills=reliable_skills, val_ids=val_ids,
                                    patience=patience, verbose=verbose)
        fit_dkt = fit_dkt_model(joint_df, train_ids, fit_pc["skill_idx"], fit_pc["type_idx"],
                                fit_pc["problem_idx"], hidden_size=dkt_hidden_size,
                                n_epochs=dkt_n_epochs, batch_size=batch_size, lr=lr,
                                seed=seed, val_ids=val_ids, patience=patience,
                                verbose=verbose)

        fit_info_rows.append(dict(run=run, n_train=len(train_ids), n_val=len(val_ids),
                                  n_test=len(test_ids),
                                  pc_best_epoch=fit_pc["best_epoch"],
                                  pc_best_val_log_loss=fit_pc["best_val_log_loss"],
                                  dkt_best_epoch=fit_dkt["best_epoch"],
                                  dkt_best_val_log_loss=fit_dkt["best_val_log_loss"]))

        eval_id_map = {"val": val_ids, "test": test_ids}
        for split_name in eval_splits:
            eval_ids = eval_id_map.get(split_name, [])
            if len(eval_ids) == 0:
                if verbose:
                    print(f"    run {run}: no {split_name} students, skipping")
                continue
            if verbose:
                print(f"    scoring on {len(eval_ids)} {split_name} students")
            diag = full_diagnostic_comparison_torch_multi(
                joint_df, skill_cluster_map, train_ids, eval_ids, fit_pc, fit_dkt,
                bkt_kwargs=bkt_kwargs, cold_threshold=cold_threshold)

            pv = pooled_vs_weighted_multi(diag["merged"], diag["per_skill"], MODEL_NAMES)
            for df_out, store in ((pv, pooled_rows),
                                  (diag["cold_warm"].copy(), cold_warm_rows),
                                  (diag["per_skill"].copy(), per_skill_rows),
                                  (diag["per_skill_cold_start"].copy(), per_skill_cs_rows),
                                  (diag["rank_corr"].copy(), rank_corr_rows)):
                df_out["run"] = run
                df_out["eval_split"] = split_name
                store.append(df_out)

    if not pooled_rows:
        raise ValueError("No run had enough overlap with this dataset's students to "
                         "evaluate -- check that student ids are formatted consistently.")

    pooled_all = pd.concat(pooled_rows, ignore_index=True)
    cold_warm_all = pd.concat(cold_warm_rows, ignore_index=True)
    per_skill_all = pd.concat(per_skill_rows, ignore_index=True)
    rank_corr_all = pd.concat(rank_corr_rows, ignore_index=True)
    per_skill_cs_all = pd.concat(per_skill_cs_rows, ignore_index=True)
    fit_info = pd.DataFrame(fit_info_rows)

    def summarize(df, group_cols, value_cols):
        agg = df.groupby(group_cols)[value_cols].agg(["mean", "std"]).reset_index()
        agg.columns = ["_".join(c).strip("_") for c in agg.columns.to_flat_index()]
        return agg

    # eval_split joins every grouping key, so mean/std are taken across RUNS within a
    # split, never across the val and test evaluations of one run.
    def value_cols(df, exclude):
        return [c for c in df.columns if c not in set(exclude) | {"run", "eval_split"}]

    summary = dict(
        pooled_vs_weighted=summarize(pooled_all, ["eval_split", "metric"],
                                     value_cols(pooled_all, ["metric"])),
        cold_warm=summarize(cold_warm_all, ["eval_split", "regime"],
                            value_cols(cold_warm_all, ["regime"])),
        per_skill=summarize(per_skill_all, ["eval_split", "skill"],
                            value_cols(per_skill_all, ["skill"])),
        per_skill_cold_start=summarize(per_skill_cs_all, ["eval_split", "skill", "regime"],
                                       value_cols(per_skill_cs_all, ["skill", "regime"])),
        rank_corr=summarize(rank_corr_all, ["eval_split", "model_a", "model_b"],
                            ["spearman_rho", "p_value", "n"]),
        fit_info=fit_info,
    )
    raw_runs = dict(pooled_vs_weighted=pooled_all, cold_warm=cold_warm_all,
                    per_skill=per_skill_all, per_skill_cold_start=per_skill_cs_all,
                    rank_corr=rank_corr_all, fit_info=fit_info)
    return summary, raw_runs

## Cross-workspace archetype transfer

Cross-workspace archetype transfer: change3 (previous lesson) -> change4.

A student's archetype belief is inferred from their change3 evidence on the skills the two
workspaces share, aligned into change4's archetype convention, and used as the starting
belief for change4. Two tests:

  static  -- zero-shot prediction for skills that exist only in change4, from the
             transferred archetype alone, never touching the new skill's own data.
  causal  -- transferred prior vs flat prior, both with full causal updating, broken down
             by opportunity index: how much the transferred belief buys on a student's
             first attempt at a skill, and how fast that advantage decays as their own
             evidence accumulates.

Archetype indices are arbitrary per fit, so the two fits are aligned by
`align_archetype_labels_across_fits` before anything is carried across.

In [ ]:
def align_archetype_labels_across_fits(W_dict_1, W_dict_2, common_student_ids) -> dict:
    """Permutation of fit 2's archetype indices best matching fit 1's.

    Hungarian assignment on the (R, R) cross-correlation matrix -- the R=2 sign-flip trick
    generalized. `perm[r1]` is fit 2's index matching fit 1's archetype r1.
    """
    ids = [s for s in common_student_ids if s in W_dict_1 and s in W_dict_2]
    if len(ids) < 10:
        raise ValueError(f"Only {len(ids)} common students with archetype beliefs in both "
                         f"fits -- too few to reliably align archetype labels.")
    W1 = np.stack([W_dict_1[s] for s in ids])
    W2 = np.stack([W_dict_2[s] for s in ids])
    R = W1.shape[1]

    corr_matrix = np.zeros((R, R))
    for r1 in range(R):
        for r2 in range(R):
            corr_matrix[r1, r2] = np.corrcoef(W1[:, r1], W2[:, r2])[0, 1]

    row_ind, col_ind = linear_sum_assignment(-corr_matrix)
    return dict(perm=col_ind, aligned_corrs=corr_matrix[row_ind, col_ind],
                n_common_students=len(ids), corr_matrix=corr_matrix)


def reindex_W_to_other_fit(W, perm):
    """W (n_students, R) in fit 1's convention -> fit 2's convention."""
    return W[:, np.argsort(perm)]


@torch.no_grad()
def causal_predictions_pc_torch_with_initial_belief(fit, joint_df, eval_ids,
                                                    W_init_dict=None) -> pd.DataFrame:
    """Keyed causal predictions, starting each student's archetype belief from
    `W_init_dict`. Students missing from it start from the population root prior, so
    `W_init_dict=None` reproduces the ordinary keyed prediction exactly."""
    padded, ids_present = build_padded_batch(fit["skill_idx"], fit["type_idx"],
                                             fit["problem_idx"], joint_df, eval_ids)
    W_init = None
    if W_init_dict is not None:
        root_prior = fit["params"].root_prior().cpu().numpy()
        W_init = np.tile(root_prior, (len(ids_present), 1))
        for i, sid in enumerate(ids_present):
            if sid in W_init_dict:
                W_init[i] = W_init_dict[sid]
    y_true, y_pred, key_rows, _ = filter_recurrence(fit, padded, key_ids=ids_present,
                                                    W_init=W_init)
    return _keyed_frame(y_true, y_pred, key_rows)


def compare_causal_transfer_vs_baseline(fit_2, joint_df_2, test_ids, W_init_dict,
                                        target_skills=None) -> pd.DataFrame:
    """The same students scored twice -- transferred prior vs population prior -- merged on
    (student_id, skill, opportunity_index), so the gap is visible attempt by attempt."""
    transferred = causal_predictions_pc_torch_with_initial_belief(
        fit_2, joint_df_2, test_ids, W_init_dict=W_init_dict)
    baseline = causal_predictions_pc_torch_with_initial_belief(
        fit_2, joint_df_2, test_ids, W_init_dict=None)
    merged = transferred.merge(
        baseline[["student_id", "skill", "opportunity_index", "y_pred"]].rename(
            columns={"y_pred": "y_pred_baseline"}),
        on=["student_id", "skill", "opportunity_index"]
    ).rename(columns={"y_pred": "y_pred_transferred"})
    if target_skills is not None:
        merged = merged[merged["skill"].isin(target_skills)]
    return merged


def causal_transfer_by_opportunity_index(merged_df, min_n=10, verbose=True) -> pd.DataFrame:
    """Transferred vs baseline aggregated by opportunity index.

    Rows below `min_n` observations are dropped: AUC on a handful of rows is sampling
    noise, and keeping them invites over-reading a noisy point as a trend.
    """
    rows = []
    for opp_idx, sub in merged_df.groupby("opportunity_index"):
        if sub["y_true"].nunique() < 2:
            continue
        y = sub["y_true"].to_numpy(float)
        m_t = compute_binary_classification_metrics(y, sub["y_pred_transferred"].to_numpy())
        m_b = compute_binary_classification_metrics(y, sub["y_pred_baseline"].to_numpy())
        rows.append(dict(opportunity_index=opp_idx, n=len(sub),
                         auc_transferred=m_t["auc"], auc_baseline=m_b["auc"],
                         log_loss_transferred=m_t["log_loss"],
                         log_loss_baseline=m_b["log_loss"]))
    result = pd.DataFrame(rows)
    before = len(result)
    result = result[result["n"] >= min_n].reset_index(drop=True) if before else result
    if verbose:
        print(f"kept {len(result)}/{before} opportunity_index rows with n >= {min_n}")
    return result


def zero_shot_transfer_to_new_skills(fit_1, joint_df_1, fit_2, joint_df_2,
                                     common_student_ids, common_skills, new_skills,
                                     test_ids, verbose=True) -> dict:
    """Predict new skills from the transferred archetype alone.

    Uses only the student's archetype belief inferred from workspace 1's evidence on the
    shared skills, plus workspace 2's own fitted cluster weights and leaves for the new
    skill's cluster -- never the new skill's own data for these students. Compared against
    workspace 2's population root prior.

    Problem type and difficulty offsets are deliberately not folded into guess/slip here.
    """
    W1_dict = compute_final_archetype_belief(fit_1, joint_df_1, common_student_ids,
                                             allowed_skills=common_skills)
    W2_dict = compute_final_archetype_belief(fit_2, joint_df_2, common_student_ids,
                                             allowed_skills=common_skills)
    alignment = align_archetype_labels_across_fits(W1_dict, W2_dict, common_student_ids)
    if verbose:
        print(f"archetype alignment: {alignment['n_common_students']} common students, "
              f"matched correlations = {alignment['aligned_corrs']}")

    test_ids_present = [s for s in test_ids if s in W1_dict]
    W1_test_aligned = reindex_W_to_other_fit(
        np.stack([W1_dict[s] for s in test_ids_present]), alignment["perm"])

    params_2 = fit_2["params"]
    with torch.no_grad():
        cwh_2 = params_2.cluster_weight_high().cpu().numpy()      # (R, C)
        leaf_2 = params_2.leaf_param().cpu().numpy()
        root_prior_2 = params_2.root_prior().cpu().numpy()
        g_logit_all = params_2.skill_guess_logit.cpu().numpy()
        s_logit_all = params_2.skill_slip_logit.cpu().numpy()

    student_to_row = {s: i for i, s in enumerate(test_ids_present)}
    rows = joint_df_2[joint_df_2["student_id"].isin(student_to_row)
                      & joint_df_2["skill"].isin(new_skills)]
    if len(rows) == 0:
        raise ValueError(
            f"No matching (student, new-skill) opportunities found -- "
            f"{len(test_ids_present)} students present. Check that new_skills' names match "
            f"joint_df_2['skill'] exactly, and that these students are in joint_df_1.")

    k_arr = rows["skill"].map(fit_2["skill_idx"]).to_numpy()
    c_arr = np.asarray(fit_2["cluster_of"])[k_arr]
    leaf_low = leaf_2[k_arr].min(axis=1)
    leaf_high = leaf_2[k_arr].max(axis=1)

    # (n_rows, R): P(this row's skill mastered | archetype r), from workspace 2's own
    # fitted cluster weights and leaves.
    cwh_rows = cwh_2.T[c_arr]
    p_mastery_per_r = cwh_rows * leaf_high[:, None] + (1 - cwh_rows) * leaf_low[:, None]
    guess_k = 1 / (1 + np.exp(-g_logit_all[k_arr]))
    slip_k = 1 / (1 + np.exp(-s_logit_all[k_arr]))
    p_correct_per_r = (p_mastery_per_r * (1 - slip_k)[:, None]
                       + (1 - p_mastery_per_r) * guess_k[:, None])
    w_rows = W1_test_aligned[rows["student_id"].map(student_to_row).to_numpy()]

    df = pd.DataFrame(dict(
        student_id=rows["student_id"].to_numpy(),
        skill=rows["skill"].to_numpy(),
        y_true=rows["correct"].astype(int).to_numpy(),
        y_pred_transferred=(w_rows * p_correct_per_r).sum(axis=1),
        y_pred_flat=(root_prior_2[None, :] * p_correct_per_r).sum(axis=1),
    ))

    y = df["y_true"].to_numpy(float)
    m_transfer = compute_binary_classification_metrics(y, df["y_pred_transferred"].to_numpy())
    m_flat = compute_binary_classification_metrics(y, df["y_pred_flat"].to_numpy())
    summary = pd.DataFrame([
        dict(model="transferred archetype", n=len(df), auc=m_transfer["auc"],
             log_loss=m_transfer["log_loss"]),
        dict(model="flat population baseline", n=len(df), auc=m_flat["auc"],
             log_loss=m_flat["log_loss"]),
    ])
    return dict(per_opportunity=df, summary=summary, alignment=alignment)


def get_valid_transfer_students(common_students, splits_1, splits_2, run) -> dict:
    """The leakage-free evaluation set for one run.

    A student must be common to both workspaces and in workspace 2's TEST split: fit_2 must
    never have seen their workspace-2 responses, or the evaluation scores data the model
    memorized. Workspace 1's split matters less -- fit_1 only *scores* the student, it is
    not evaluated on them -- but `valid_strict` restricts to its test split too.

    Validation students are excluded from both: the PC chose its stopping epoch on them.
    """
    train_ids_1, val_ids_1, test_ids_1 = as_three_way(splits_1[run])
    train_ids_2, val_ids_2, test_ids_2 = as_three_way(splits_2[run])
    common_set = set(common_students)
    valid_soft = sorted(common_set & set(test_ids_2))
    valid_strict = sorted(common_set & set(test_ids_2) & set(test_ids_1))
    return dict(valid_soft=valid_soft, valid_strict=valid_strict,
                train_ids_1=train_ids_1, val_ids_1=val_ids_1,
                train_ids_2=train_ids_2, val_ids_2=val_ids_2,
                n_common=len(common_set), n_valid_soft=len(valid_soft),
                n_valid_strict=len(valid_strict))


def causal_transfer_test_for_run(fit_1, joint_1, fit_2, joint_2, eval_students,
                                 common_skills, perm_R, min_n=None):
    """Aligned transfer-vs-baseline for one run. -> (merged, by_opportunity, W_init_dict)."""
    W1_dict_full = compute_final_archetype_belief(fit_1, joint_1, eval_students,
                                                  allowed_skills=common_skills)
    W_init_dict = {sid: reindex_W_to_other_fit(np.array(w).reshape(1, -1), perm_R)[0]
                   for sid, w in W1_dict_full.items()}
    merged = compare_causal_transfer_vs_baseline(fit_2, joint_2, eval_students, W_init_dict,
                                                 target_skills=common_skills)
    if min_n is None:
        min_n = max(10, len(eval_students) // 10)
    return merged, causal_transfer_by_opportunity_index(merged, min_n=min_n), W_init_dict


def run_leakage_free_transfer_evaluation_full(joint_1, joint_2, skill_cluster_map_1,
                                              skill_cluster_map_2, splits_1, splits_2,
                                              common_skills, new_skills, n_epochs=20,
                                              R=2, L=2, batch_size=128, lr=0.01,
                                              patience=None, use_strict=False,
                                              verbose=True):
    """Both transfer tests, per run, on the same refits.

    -> (summary_df, causal_merged_all, by_opportunity_all). The last is the mean AUC (+- sd)
    for new skills grouped by opportunity, once averaged across runs.
    """
    shared_runs = sorted(set(splits_1) & set(splits_2))
    common_students = sorted(set(joint_1["student_id"]) & set(joint_2["student_id"]))
    summary_rows, all_causal_merged, all_by_opportunity = [], [], []

    for run in shared_runs:
        train_ids_1, val_ids_1, _ = as_three_way(splits_1[run])
        train_ids_2, val_ids_2, _ = as_three_way(splits_2[run])

        student_info = get_valid_transfer_students(common_students, splits_1, splits_2, run)
        eval_students = (student_info["valid_strict"] if use_strict
                         else student_info["valid_soft"])
        if verbose:
            print(f"--- run {run}: {len(eval_students)} leakage-free eval students "
                  f"(of {student_info['n_common']} common; strict would give "
                  f"{student_info['n_valid_strict']}) ---")
        if len(eval_students) < 10:
            print(f"  SKIPPING run {run}: too few leakage-free students "
                  f"({len(eval_students)})")
            continue

        fit_1 = fit_pc_model_torch(joint_1, skill_cluster_map_1, train_ids_1, R=R, L=L,
                                   n_epochs=n_epochs, batch_size=batch_size, lr=lr,
                                   val_ids=val_ids_1, patience=patience, verbose=verbose)
        fit_2 = fit_pc_model_torch(joint_2, skill_cluster_map_2, train_ids_2, R=R, L=L,
                                   n_epochs=n_epochs, batch_size=batch_size, lr=lr,
                                   val_ids=val_ids_2, patience=patience, verbose=verbose)

        W1_dict = compute_final_archetype_belief(fit_1, joint_1, eval_students,
                                                 allowed_skills=common_skills)
        W2_dict = compute_final_archetype_belief(fit_2, joint_2, eval_students,
                                                 allowed_skills=common_skills)
        alignment = align_archetype_labels_across_fits(W1_dict, W2_dict, eval_students)

        static_summary = zero_shot_transfer_to_new_skills(
            fit_1, joint_1, fit_2, joint_2, eval_students, common_skills, new_skills,
            test_ids=eval_students, verbose=verbose)["summary"].set_index("model")

        causal_merged, by_opportunity, _ = causal_transfer_test_for_run(
            fit_1, joint_1, fit_2, joint_2, eval_students, common_skills,
            alignment["perm"])
        all_causal_merged.append(causal_merged.assign(run=run))
        all_by_opportunity.append(by_opportunity.assign(run=run))

        def _metrics(sub, col):
            if not len(sub) or sub["y_true"].nunique() < 2:
                return {"auc": np.nan, "log_loss": np.nan}
            return compute_binary_classification_metrics(sub["y_true"].to_numpy(float),
                                                         sub[col].to_numpy())

        def _static(model, metric):
            return (static_summary.loc[model, metric] if model in static_summary.index
                    else np.nan)

        opp0 = causal_merged[causal_merged["opportunity_index"] == 0]
        m_t0, m_b0 = _metrics(opp0, "y_pred_transferred"), _metrics(opp0, "y_pred_baseline")
        m_t = _metrics(causal_merged, "y_pred_transferred")
        m_b = _metrics(causal_merged, "y_pred_baseline")

        summary_rows.append(dict(
            run=run, n_eval_students=len(eval_students),
            alignment_corr=alignment["aligned_corrs"][0],
            static_auc_transferred=_static("transferred archetype", "auc"),
            static_auc_flat=_static("flat population baseline", "auc"),
            static_log_loss_transferred=_static("transferred archetype", "log_loss"),
            static_log_loss_flat=_static("flat population baseline", "log_loss"),
            causal_opp0_auc_transferred=m_t0["auc"], causal_opp0_auc_baseline=m_b0["auc"],
            causal_opp0_log_loss_transferred=m_t0["log_loss"],
            causal_opp0_log_loss_baseline=m_b0["log_loss"],
            causal_pooled_auc_transferred=m_t["auc"],
            causal_pooled_auc_baseline=m_b["auc"],
            causal_pooled_log_loss_transferred=m_t["log_loss"],
            causal_pooled_log_loss_baseline=m_b["log_loss"],
        ))

    return (pd.DataFrame(summary_rows),
            pd.concat(all_causal_merged, ignore_index=True) if all_causal_merged else pd.DataFrame(),
            pd.concat(all_by_opportunity, ignore_index=True) if all_by_opportunity else pd.DataFrame())




def mean_auc_by_opportunity(by_opportunity_all: pd.DataFrame) -> pd.DataFrame:
    """Mean AUC (+- sd) across runs, grouped by opportunity index -- the headline transfer
    table: what the transferred archetype buys on a new skill's first attempt, and how fast
    it decays."""
    if by_opportunity_all.empty:
        return by_opportunity_all
    value_cols = [c for c in by_opportunity_all.columns
                  if c not in {"opportunity_index", "run"}]
    agg = (by_opportunity_all.groupby("opportunity_index")[value_cols]
           .agg(["mean", "std"]).reset_index())
    agg.columns = ["_".join(c).strip("_") for c in agg.columns.to_flat_index()]
    return agg

## Archetype interpretation and figures

What the PC learned, and where each student sits in it.

Two things a fitted circuit exposes that the baselines cannot: the (R, C)
`cluster_weight_high` matrix -- P(high local state | archetype, cluster), i.e. what each
global archetype means in terms of skill clusters -- and each student's posterior over
those archetypes.

In [ ]:
def archetype_log_odds(w0, eps=1e-12):
    """log P(archetype=0) / P(archetype=1), for R=2.

    Undoes sigmoid's compression near 0 and 1, so confidence levels that look identical in
    raw probability (1e-6 vs 1e-3) are correctly spread apart.
    """
    w0 = np.clip(w0, eps, 1 - eps)
    return np.log(w0 / (1 - w0))


def archetype_belief_frame(W_dict) -> pd.DataFrame:
    """student -> P(archetype=0) and its log-odds."""
    rows = [dict(student_id=sid, p_archetype_0=float(np.asarray(w)[0]),
                 log_odds=float(archetype_log_odds(np.asarray(w)[0])))
            for sid, w in W_dict.items()]
    return pd.DataFrame(rows).sort_values("log_odds").reset_index(drop=True)


def plot_archetype_belief_distribution(W_dict, save_path=None, title=""):
    """The student archetype-belief distribution, in log-odds.

    Log-odds rather than raw probability: sigmoid compresses near 0 and 1, so a smooth,
    evenly spread distribution looks like a spike at the extremes once compressed.
    """
    log_odds_values = archetype_log_odds(
        np.array([np.asarray(w)[0] for w in W_dict.values()]))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(log_odds_values, bins=40, color="#DD8452", edgecolor="black", linewidth=0.5)
    ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_xlabel("log-odds: log(P(archetype=0) / P(archetype=1))")
    if title:
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved {save_path}")
    return fig


def plot_cluster_weight_high_heatmap(cwh_mean, cwh_sd=None, cluster_names=None,
                                     save_path=None, title=""):
    """Rows are global archetypes, columns skill clusters, cells P(local state = high).

    Annotated with mean ± sd across fits: R and C are both small and the numbers are the
    point of the figure.
    """
    cwh_mean = np.asarray(cwh_mean, dtype=float)
    R, C = cwh_mean.shape
    if cluster_names is None:
        cluster_names = [f"cluster {c}" for c in range(C)]
    if len(cluster_names) != C:
        raise ValueError(f"got {len(cluster_names)} cluster names for {C} columns")

    fig, ax = plt.subplots(figsize=(max(6.0, 1.1 * C + 1.5), max(3.2, 1.6 * R + 1.6)))
    im = ax.imshow(cwh_mean, cmap="RdYlGn", vmin=0.0, vmax=1.0, aspect="auto")
    ax.set_xticks(range(C))
    ax.set_xticklabels(cluster_names, rotation=30, ha="right")
    ax.set_yticks(range(R))
    ax.set_yticklabels([f"archetype {r}" for r in range(R)])
    if title:
        ax.set_title(title, fontsize=10)

    for r in range(R):
        for c in range(C):
            value = cwh_mean[r, c]
            text = (f"{value:.2f}" if cwh_sd is None
                    else f"{value:.2f}\n$\\pm${np.asarray(cwh_sd)[r, c]:.2f}")
            # white on the saturated ends of RdYlGn, black through its pale middle
            ax.text(c, r, text, ha="center", va="center", fontsize=10,
                    color="white" if (value < 0.30 or value > 0.70) else "black")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("P(local state = high)")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved {save_path}")
    return fig


def cluster_weight_high_across_runs(fits, joint_df, student_ids, verbose=True) -> dict:
    """Mean and sd of cluster_weight_high across several fits of the same workspace.

    Archetype indices are arbitrary per fit, so every fit is first aligned to the first
    one via its students' beliefs. Averaging without that step would mix archetype 0 of one
    run with archetype 1 of another and inflate the sd into meaninglessness.
    """

    beliefs = [compute_final_archetype_belief(f, joint_df, student_ids) for f in fits]
    matrices = []
    for i, fit in enumerate(fits):
        with torch.no_grad():
            cwh = fit["params"].cluster_weight_high().cpu().numpy()
        if i == 0:
            matrices.append(cwh)
            continue
        alignment = align_archetype_labels_across_fits(beliefs[0], beliefs[i], student_ids)
        matrices.append(cwh[alignment["perm"]])
        if verbose:
            print(f"  run {i} aligned to run 0 (r = {alignment['aligned_corrs'][0]:.3f})")

    stack = np.stack(matrices)
    return dict(mean=stack.mean(axis=0), sd=stack.std(axis=0, ddof=0),
                clusters=fits[0]["clusters"], n_fits=len(fits), beliefs=beliefs[0])




def students_at_log_odds_extremes(W_dict, n_per_end=5) -> dict:
    """The `n_per_end` students most confidently in each archetype.

    Log-odds is log P(archetype=0) / P(archetype=1), so the most *positive* students are the
    ones the model is most sure belong to archetype 0, and the most negative to archetype 1.
    Returned as id lists, to rebuild those students' raw interactions and inspect how the
    two ends actually differ.
    """
    frame = archetype_belief_frame(W_dict)      # ascending log-odds
    return dict(archetype_1=frame.head(n_per_end),
                archetype_0=frame.tail(n_per_end).iloc[::-1],
                frame=frame)

## Parameters

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────────────
RUNS = 5              # independent school-stratified split replicates
EPOCHS = 20           # Adam budget; the best-validating epoch is kept
R_ARCHETYPES = 2      # R, global archetypes
L_STATES = 2          # L, per-cluster local states
BATCH_SIZE = 128
LR = 0.01
TRAIN_FRAC, VAL_FRAC = 0.60, 0.10
SEED = 0
N_EXTREME = 10        # students listed at each end of the archetype log-odds distribution

# Set to e.g. 500 for a fast end-to-end check. Whole schools are sampled, not individual
# students: a scattered sample leaves most schools with one or two students, and the
# stratified split sends those entirely to train, starving val and test.
MAX_STUDENTS = None

SOURCE = "ratio_proportion_change3"   # PercentageChange, the previous lesson
TARGET = "ratio_proportion_change4"   # PercentageUse, where the new skills are
DATASET_LABEL = {TARGET: "dataset1", SOURCE: "dataset2"}

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"figures -> {FIGURES_DIR}")

## Loading a workspace

In [ ]:
def load_ws(name, verbose=True):
    """data.pkl + problem_info.csv -> (joint table, student->school map, cluster map)."""
    ws_dir = workspace_dir(name)
    if not (ws_dir / "data.pkl").exists():
        raise FileNotFoundError(f"{ws_dir}/data.pkl not found -- run notebook 1 first.")
    cluster_map, subset = for_workspace(name)
    loaded = load_workspace(str(ws_dir), name, verbose=verbose)
    report_skill_coverage(loaded["joint"], cluster_map, name, verbose=verbose)
    joint_df = apply_skill_subset(loaded["joint"], cluster_map, skills_to_include=subset,
                                  name=name, verbose=verbose)
    return joint_df, loaded["student_school_map"], cluster_map

## Figures — the learned archetypes, and where students sit in them

In [ ]:
def archetype_figures(name, verbose=True):
    """The two per-workspace figures: the learned archetypes, and where students sit."""
    joint_df, school_map_, cluster_map = load_ws(name, verbose=verbose)
    students = sorted(joint_df["student_id"].unique())
    if MAX_STUDENTS:
        students = subsample_by_school(students, school_map_, MAX_STUDENTS, seed=SEED,
                                       verbose=verbose)
        joint_df = joint_df[joint_df["student_id"].isin(set(students))]

    fits = []
    for run in range(RUNS):
        train_ids, val_ids, _ = stratified_school_split(
            students, school_map_, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC,
            seed=SEED + run, verbose=verbose)
        print(f"--- {name} fit {run} ---")
        fits.append(fit_pc_model_torch(
            joint_df, cluster_map, train_ids, R=R_ARCHETYPES, L=L_STATES,
            n_epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=SEED + run,
            val_ids=val_ids, verbose=verbose))

    label = DATASET_LABEL[name]
    agg = cluster_weight_high_across_runs(fits, joint_df, students, verbose=verbose)
    plot_cluster_weight_high_heatmap(
        agg["mean"], cwh_sd=agg["sd"], cluster_names=agg["clusters"],
        save_path=FIGURES_DIR / f"{label}-cwh-mean.png")
    plot_archetype_belief_distribution(
        agg["beliefs"], save_path=FIGURES_DIR / f"{label}-archetype-distribution.png")
    plt.show()

    # The students the model is most certain about, at each end. Feed these ids to
    # notebook 2 to rebuild their raw interactions.
    ends = students_at_log_odds_extremes(agg["beliefs"], n_per_end=N_EXTREME)
    print(f"\nStudents at the archetype log-odds extremes -- {name}")
    for archetype, key in ((0, "archetype_0"), (1, "archetype_1")):
        rows = ends[key]
        print(f"\n  most confident archetype {archetype} "
              f"(log-odds {rows['log_odds'].iloc[0]:+.2f} to {rows['log_odds'].iloc[-1]:+.2f}):")
        print(f"    {rows['student_id'].tolist()}")
    return agg, ends


extremes = {}
for _name in (TARGET, SOURCE):
    print(f"\n{'=' * 70}\n{_name}  ({DATASET_LABEL[_name]})\n{'=' * 70}")
    _agg, extremes[_name] = archetype_figures(_name)

## Table 2 — archetype transfer, change3 -> change4

Evaluation students must be in change4's test split, so the model being scored never trained on them. Archetype indices are arbitrary per fit, so the two fits are aligned by Hungarian assignment before any belief is carried across.

In [ ]:
joint_src, schools_src, cmap_src = load_ws(SOURCE)
joint_tgt, schools_tgt, cmap_tgt = load_ws(TARGET)

common = sorted(set(joint_src["student_id"]) & set(joint_tgt["student_id"]))
print(f"\n{len(common)} students appear in both workspaces -- transfer keys on these")
if not common:
    raise ValueError("no students in common; transfer needs the same students in both.")

if MAX_STUDENTS:
    # Sample from the COMMON set and restrict both: sampling each workspace independently
    # would shrink the overlap transfer depends on.
    common = subsample_by_school(common, schools_src, MAX_STUDENTS, seed=SEED)
    joint_src = joint_src[joint_src["student_id"].isin(set(common))]
    joint_tgt = joint_tgt[joint_tgt["student_id"].isin(set(common))]

check_zero_shot(joint_src)

splits_src = generate_school_stratified_splits(
    sorted(joint_src["student_id"].unique()), schools_src, RUNS,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, base_seed=SEED, verbose=False)
splits_tgt = generate_school_stratified_splits(
    sorted(joint_tgt["student_id"].unique()), schools_tgt, RUNS,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, base_seed=SEED, verbose=False)

summary, causal_merged, by_opportunity = run_leakage_free_transfer_evaluation_full(
    joint_src, joint_tgt, cmap_src, cmap_tgt, splits_src, splits_tgt,
    common_skills=COMMON_SKILLS, new_skills=NEW_SKILLS, n_epochs=EPOCHS,
    R=R_ARCHETYPES, L=L_STATES, batch_size=BATCH_SIZE, lr=LR, verbose=True)

if summary.empty:
    raise ValueError("every run was skipped -- too few leakage-free students.")

by_opportunity_mean = mean_auc_by_opportunity(by_opportunity)

MAX_OPPORTUNITY = 5      # the range the paper's table reports

rows = by_opportunity_mean[
    by_opportunity_mean["opportunity_index"] <= MAX_OPPORTUNITY]

print("\nMean AUC (+- sd) for new skills in PercentageUse, transferring archetype")
print("beliefs from the previous lesson (PercentageChange), by opportunity.")
print(f"{RUNS} split replicates, {len(common):,} students common to both workspaces.\n")
print(f"{'Opportunity':>12}  {'Transferred':>17}  {'Baseline':>17}")
print(f"{'-' * 12}  {'-' * 17}  {'-' * 17}")
for r in rows.itertuples():
    print(f"{int(r.opportunity_index):>12}  "
          f"{r.auc_transferred_mean:>9.3f} +- {r.auc_transferred_std:<5.3f}  "
          f"{r.auc_baseline_mean:>9.3f} +- {r.auc_baseline_std:<5.3f}")

print("\nFull range, with log loss:")
print(by_opportunity_mean.to_string(index=False))

## PC vs BKT vs DKT

Students, not opportunities, are split three ways. TRAIN fits every model; VAL picks the stopping epoch for the two gradient-fit models and is used for nothing else; TEST is scored and never touches fitting or model selection. BKT has no stopping knob and ignores VAL, so all three are still compared on identical TEST opportunities. Every model emits predictions keyed by `(student_id, skill, opportunity_index)` and the keys are inner-joined, so each number below covers exactly the same rows.

In [ ]:
# The most expensive cell here: BKT is fit per skill, and PC and DKT are each fit once per
# replicate. Reduce RUNS / EPOCHS above if you only want to watch it run.
COLD_THRESHOLD = 3      # opportunities below this count as a student's cold start

joint_cmp, schools_cmp, cmap_cmp = load_ws(TARGET)
students_cmp = sorted(joint_cmp["student_id"].unique())
if MAX_STUDENTS:
    students_cmp = subsample_by_school(students_cmp, schools_cmp, MAX_STUDENTS, seed=SEED)
    joint_cmp = joint_cmp[joint_cmp["student_id"].isin(set(students_cmp))]

splits_cmp = generate_school_stratified_splits(
    students_cmp, schools_cmp, RUNS, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC,
    base_seed=SEED, verbose=False)

summary_cmp, runs_cmp = multi_run_school_stratified_evaluation_three_way(
    joint_cmp, cmap_cmp, splits_cmp, R=R_ARCHETYPES, L=L_STATES, pc_n_epochs=EPOCHS,
    dkt_n_epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=SEED,
    cold_threshold=COLD_THRESHOLD, eval_splits=("test",), verbose=True,
    verbose_filter=False)


def _row(label, values):
    """One printed line: a label and three mean +- sd cells."""
    cells = "  ".join(
        (f"{m:>7.3f} +- {sd:<5.3f}" if pd.notna(sd) else f"{m:>7.3f}         ")
        for m, sd in values)
    print(f"{label:>18}  {cells}")


def _header():
    print(f"{'':>18}  {'PC':>17}  {'BKT':>17}  {'DKT':>17}")
    print(f"{'-' * 18}  {'-' * 17}  {'-' * 17}  {'-' * 17}")


pooled = summary_cmp["pooled_vs_weighted"]
pooled = pooled[pooled["eval_split"] == "test"]
print(f"\nPC vs BKT vs DKT on {TARGET}")
print(f"{RUNS} replicates, held-out test students, identical opportunities for all three.\n")
_header()
for r in pooled.itertuples():
    _row(r.metric, [(r.pc_pooled_mean, r.pc_pooled_std),
                    (r.bkt_pooled_mean, r.bkt_pooled_std),
                    (r.dkt_pooled_mean, r.dkt_pooled_std)])

cold_warm = summary_cmp["cold_warm"]
cold_warm = cold_warm[cold_warm["eval_split"] == "test"]
print(f"\nAUC by regime. Cold is a student's first {COLD_THRESHOLD} opportunities on a skill,")
print("where BKT has only its flat prior while the circuit can borrow evidence from")
print("correlated skills.\n")
_header()
for r in cold_warm.itertuples():
    _row(r.regime, [(r.auc_pc_mean, r.auc_pc_std),
                    (r.auc_bkt_mean, r.auc_bkt_std),
                    (r.auc_dkt_mean, r.auc_dkt_std)])